# Проект 1. Почему каналы различаются по росту выручки и удержанию

**Автор:** Шевченко Андрей, 262. **Основной период:** март–июнь 2026, UTC.

Сравниваем все каналы привлечения. После общего обзора подробно рассматриваем
social относительно остальных каналов вместе. Social выбран по исходному
наблюдению о слабом росте; это исследовательское сравнение, а не заранее
зарегистрированная статистическая проверка. Состав остальных каналов сохраняется,
их показатели объединяются через числители и знаменатели, а не среднее процентов.

## 1. Первичный анализ выручки и постановка бизнес-вопроса

Выручка рассчитывается как сумма платежей со статусом `success` по дате платежа.
Платежи `failed` и `refunded` исключаются. Повторного вычитания возвратов нет.
Показатель отражает успешные списания за период и не является прибылью.

Канал определяется по владельцу подписки. Участники не присоединяются к платежам,
поэтому совместное использование не увеличивает сумму выручки при соединении.
Отсутствующий канал сохраняется отдельной категорией. История изменения канала
не восстанавливается.

Расчёт выполняется на уровне месяца и канала. Месячная детализация позволяет сравнивать
динамику выручки с движением базы подписок. Для сравнения месяцев разной длины
рассчитывается средняя дневная выручка. Дневные данные используются для проверки
полноты, а наблюдения на уровне подписки и месяца используются для расчёта оттока.
Одна подписка может присутствовать в нескольких месяцах.

В основном сравнении рассматриваются март и июнь 2026 года: оба месяца целиком лежат после
мартовской смены записей цен и не содержат обнаруженного апрельского разрыва.
Справочник тарифов неоднозначен: у расширенного тарифа запись 429 ₽ за февраль–июнь
перекрывает другие интервалы. Поэтому изменение цен не восстанавливаем простым
соединением со справочником. Ниже выводятся записи и пересечения; выручка и средний
платёж считаются по фактическим успешным списаниям. Дополнительное сравнение января и июня
охватывает изменение среднего платежа, но не оценивает чистый эффект цены.
Нормирование на дни не устраняет сезонность и изменение состава подписок.

За 14 и 15 апреля отсутствуют платежи и записи использования сервисов.
Апрельские суммы не восстанавливаются и не используются как конечная точка
факторного сравнения. Наличие событий за день само по себе не гарантирует
полноту источника.

Для воспроизведения используются зависимости из `pyproject.toml` и `uv.lock`,
подключение через `.env` и последовательное выполнение ячеек из корня проекта
или папки `notebooks`. После запуска агрегаты сохраняются в отдельный каталог запуска внутри
`reports/project1_revised`; старые отчёты не используются как входные данные.


In [ ]:
from pathlib import Path
from itertools import permutations
import sys
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
from sqlalchemy import text

ROOT = Path.cwd()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent
assert (ROOT / "src").exists(), "Откройте ноутбук из корня проекта или notebooks"
sys.path.insert(0, str(ROOT / "src"))
from prime.config import get_engine
engine = get_engine()

def q(sql, **params):
    """Выполнить запрос к базе данных и вернуть таблицу."""
    with engine.connect() as conn:
        conn.execute(text("set transaction read only"))
        conn.execute(text("set time zone 'UTC'"))
        return pd.read_sql(text(sql), conn, params=params)

MONTHS = {1: "Январь", 2: "Февраль", 3: "Март", 4: "Апрель", 5: "Май", 6: "Июнь"}
PERIODS = pd.date_range("2026-01-01", "2026-06-01", freq="MS")
PERIOD_PAIRS = [(pd.Timestamp("2026-03-01"), pd.Timestamp("2026-06-01")),
                (pd.Timestamp("2026-01-01"), pd.Timestamp("2026-06-01"))]
GROUPS = ["crm", "organic", "partner_telecom", "performance", "referral", "social", "без канала"]
plt.rcParams.update({"font.size": 10, "axes.spines.top": False, "axes.spines.right": False})

FOCUS = "social"
pd.set_option("display.max_colwidth", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 120)
pd.set_option("display.float_format", lambda x: f"{x:,.3f}".replace(",", " "))

def check_base_totals(data):
    """Каждый разрез должен сохранять подписко-месяцы и завершения."""
    actual = data.groupby(["month", "channel"])[["base_start", "churned"]].sum().sort_index()
    expected = base.rename(columns={"comparison": "channel"}).set_index(
        ["month", "channel"])[["base_start", "churned"]].sort_index()
    pd.testing.assert_frame_equal(actual, expected, check_dtype=False)

def waterfall(deltas, title, ylabel="тыс. ₽ в день", subtitle=""):
    """Каскад изменений из С6: накопленные начала, вклады и итог от нуля."""
    deltas = deltas.astype(float)
    assert len(deltas) and np.isfinite(deltas).all()
    before = deltas.cumsum() - deltas
    after = before + deltas
    positions = np.arange(len(deltas))
    fig, ax = plt.subplots(figsize=(11, 5.5), layout="constrained")
    colors = np.where(deltas < 0, "#C1272D", "#2A9D8F")
    # Отрицательный вклад идёт вниз от предыдущего накопленного уровня.
    ax.bar(positions, deltas, bottom=before, color=colors, width=.65, zorder=2)
    ax.bar(len(deltas), deltas.sum(), color="#6B7280", width=.65, zorder=2)
    for i, (v, end) in enumerate(zip(deltas, after)):
        ax.annotate(f"{v:+.2f}", (i, end), xytext=(0, 5 if v >= 0 else -5),
                    textcoords="offset points", ha="center",
                    va="bottom" if v >= 0 else "top")
        ax.plot([i + .325, i + .675], [end, end], color="#9CA3AF", linewidth=.8)
    total = deltas.sum()
    ax.annotate(f"{total:+.2f}", (len(deltas), total),
                xytext=(0, 5 if total >= 0 else -5), textcoords="offset points",
                ha="center", va="bottom" if total >= 0 else "top")
    ax.set_xticks(np.arange(len(deltas) + 1), [*deltas.index, "Итого"])
    ax.set(title=title, ylabel=ylabel, xlabel=subtitle)
    ax.axhline(0, color="#111827", linewidth=.8)
    ax.grid(axis="y", color="#E5E7EB")
    ax.set_axisbelow(True)
    levels = np.r_[0, before.to_numpy(), after.to_numpy(), total]
    margin = max(np.ptp(levels) * .22, .1)
    ax.set_ylim(levels.min() - margin, levels.max() + margin)
    return ax


### 1.1. Качество и полнота данных

Проверяются уникальность ключей, сохранность соединений, положительность сумм
и временные границы подписок. Платежи вне срока учитываются отдельно.
Основной расчет сохраняет все успешные платежи. Анализ чувствительности
показывает результат при исключении платежей вне `[started_at, ended_at)`
и близких повторов.

Разные идентификаторы платежей не исключают повторного списания. Поэтому
дополнительно проверяется близость платежей одной суммы по одной подписке
во времени. Достоверность подозрительного списания требует подтверждения
по данным платежного процесса.

In [ ]:
payment_calendar = q("""
    select paid_at::date as day, count(*) as attempts
    from prime.payments
    where paid_at >= '2026-01-01'::timestamptz
      and paid_at < '2026-08-01'::timestamptz
    group by 1 order by 1
""")
payment_calendar["day"] = pd.to_datetime(payment_calendar["day"])
payment_days = pd.DatetimeIndex(payment_calendar["day"])
calendar = pd.date_range("2026-01-01", "2026-07-31", freq="D")
missing_days = calendar.difference(payment_days)
coverage_table = pd.DataFrame({
    "Месяц": [MONTHS[m.month] for m in PERIODS],
    "Календарных дней": [m.days_in_month for m in PERIODS],
    "Дней без записей": [sum((missing_days >= m) & (missing_days < m + pd.offsets.MonthBegin(1))) for m in PERIODS]
}).set_index("Месяц")


PAYMENT_CTE = """
with tagged as (
    select p.payment_id, p.subscription_id, p.paid_at, p.amount,
           s.subscription_id is not null as has_subscription,
           c.client_id is not null as has_owner,
           coalesce(c.acquisition_channel, 'без канала') as comparison,
           p.paid_at < s.started_at as before_start,
           p.paid_at >= s.ended_at as after_end,
           p.paid_at >= s.started_at
             and (s.ended_at is null or p.paid_at < s.ended_at) as within_term
    from prime.payments p
    left join prime.subscriptions s on s.subscription_id = p.subscription_id
    left join prime.clients c on c.client_id = s.owner_client_id
    where p.status = 'success'
      and p.paid_at >= '2026-01-01'::timestamptz
      and p.paid_at < '2026-07-01'::timestamptz
)
"""
audit = q(PAYMENT_CTE + """
select count(*) as success_payments, sum(amount) as revenue,
       count(*) filter (where not has_subscription) as missing_subscription,
       count(*) filter (where not has_owner) as missing_owner,
       count(*) filter (where amount is null or amount <= 0) as bad_amount,
       count(*) filter (where before_start) as before_start,
       count(*) filter (where after_end) as after_end
from tagged
""")
display(coverage_table)
display(audit.T)
assert audit[["missing_subscription", "missing_owner", "bad_amount"]].eq(0).all().all()

key_check = q("""
select 'Платежи' as entity, count(*) as rows, count(distinct payment_id) as ids
from prime.payments where paid_at >= '2026-01-01' and paid_at < '2026-07-01'
union all
select 'Подписки', count(*), count(distinct subscription_id) from prime.subscriptions
union all
select 'Клиенты', count(*), count(distinct client_id) from prime.clients
""")
assert key_check["rows"].eq(key_check["ids"]).all()
subscription_quality = q("""
select count(*) filter (where started_at is null) as missing_start,
       count(*) filter (where ended_at < started_at) as reversed_dates
from prime.subscriptions
""")
assert subscription_quality.eq(0).all().all()

# Два базовых правила; вариант без близких повторов добавляется следующей ячейкой.
cash = q(PAYMENT_CTE + """
select date_trunc('month', paid_at)::date as month, comparison,
       'Все успешные' as policy, count(*) as payments, sum(amount) as revenue
from tagged
group by 1, 2
union all
select date_trunc('month', paid_at)::date as month, comparison,
       'Только в сроке' as policy, count(*) as payments, sum(amount) as revenue
from tagged
where within_term
group by 1, 2
order by 1, 2, 3
""")
cash["month"] = pd.to_datetime(cash["month"])
assert not cash.duplicated(["month", "comparison", "policy"]).any()
assert set(cash["comparison"]) == set(GROUPS)
assert np.isclose(cash.loc[cash["policy"].eq("Все успешные"), "revenue"].sum(), audit.loc[0, "revenue"])


tariff_history = q("""
select tariff_id, tariff_name, monthly_fee, valid_from, valid_to
from prime.tariffs order by tariff_id, valid_from, monthly_fee
""")
tariff_overlaps = q("""
with numbered as (
    select *, row_number() over (
        order by tariff_id, valid_from, valid_to, monthly_fee) as row_id
    from prime.tariffs
)
select a.tariff_id, a.monthly_fee as price_a, b.monthly_fee as price_b,
       greatest(a.valid_from, b.valid_from) as overlap_start,
       least(a.valid_to, b.valid_to) as overlap_end
from numbered a join numbered b on a.tariff_id = b.tariff_id and a.row_id < b.row_id
where a.valid_from <= b.valid_to and b.valid_from <= a.valid_to
order by 1, 4
""")
display(tariff_history)
display(tariff_overlaps)


#### Близкие повторы и существенность аномалий

Подозрительным повтором считается второй или последующий успешный платёж той же суммы
по той же подписке менее чем через заданное число минут. Соседний платёж
определяется SQL-функцией `LAG` (аналог `groupby().shift()`); первые платежи
цепочек сохраняются. Загружаются только агрегаты, а не миллион исходных строк.
Для границы января учитываются последние 20 минут декабря.

Пороги 10, 15 и 20 минут показывают чувствительность определения; они не доказывают
природу повторного списания. Для расчётов используется порог 20 минут.
Отдельно оцениваются сумма платежей вне срока и сумма близких повторов.
Пересечение этих аномалий учтено в четвёртом правиле: внутри срока и без повторов.


In [ ]:
DUPLICATE_CTE = """
with ordered as (
    select p.payment_id, p.subscription_id, p.paid_at, p.amount,
           coalesce(c.acquisition_channel, 'без канала') as comparison,
           p.paid_at >= s.started_at
             and (s.ended_at is null or p.paid_at < s.ended_at) as within_term,
           lag(p.paid_at) over (partition by p.subscription_id, p.amount
               order by p.paid_at, p.payment_id) as previous_at
    from prime.payments p
    join prime.subscriptions s using(subscription_id)
    join prime.clients c on c.client_id = s.owner_client_id
    where p.status = 'success'
      and p.paid_at >= '2025-12-31 23:40:00+00'::timestamptz
      and p.paid_at < '2026-07-01'::timestamptz
), tagged as (
    select *, coalesce(paid_at - previous_at < interval '20 minutes', false) as is_duplicate
    from ordered where paid_at >= '2026-01-01'::timestamptz
)
"""
duplicate_summary = q(DUPLICATE_CTE + """
select threshold as minutes,
       count(*) filter (where paid_at - previous_at < threshold * interval '1 minute') as duplicate_payments,
       count(distinct subscription_id) filter (
           where paid_at - previous_at < threshold * interval '1 minute') as subscriptions,
       coalesce(sum(amount) filter (
           where paid_at - previous_at < threshold * interval '1 minute'), 0) as duplicate_amount
from tagged cross join (values (10), (15), (20)) t(threshold)
group by 1 order by 1
""")
duplicate_summary["Доля выручки, %"] = duplicate_summary["duplicate_amount"] / audit.loc[0, "revenue"] * 100
duplicate_by_channel = q(DUPLICATE_CTE + """
select comparison, count(*) as payments, sum(amount) as revenue,
       count(*) filter (where is_duplicate) as duplicate_payments,
       coalesce(sum(amount) filter (where is_duplicate), 0) as duplicate_amount,
       count(*) filter (where not within_term) as out_of_term_payments,
       coalesce(sum(amount) filter (where not within_term), 0) as out_of_term_amount
from tagged group by 1 order by 1
""").set_index("comparison").reindex(GROUPS)
assert np.isclose(duplicate_by_channel["revenue"].sum(), audit.loc[0, "revenue"], atol=.01, rtol=0)
for field in ["duplicate", "out_of_term"]:
    duplicate_by_channel[field + "_revenue_pct"] = (
        duplicate_by_channel[field + "_amount"] / duplicate_by_channel["revenue"] * 100)
clean_cash = q(DUPLICATE_CTE + """
select date_trunc('month', paid_at)::date as month, comparison,
       'Без повторов 20 минут' as policy, count(*) as payments, sum(amount) as revenue
from tagged where not is_duplicate group by 1, 2
union all
select date_trunc('month', paid_at)::date as month, comparison,
       'В сроке и без повторов' as policy, count(*) as payments, sum(amount) as revenue
from tagged where not is_duplicate and within_term group by 1, 2
""")
clean_cash["month"] = pd.to_datetime(clean_cash["month"])
assert np.isclose(clean_cash.loc[clean_cash["policy"].eq("Без повторов 20 минут"), "revenue"].sum(),
                  audit.loc[0, "revenue"] - duplicate_by_channel["duplicate_amount"].sum(), atol=.01, rtol=0)
cash = pd.concat([cash, clean_cash], ignore_index=True)
assert not cash.duplicated(["month", "comparison", "policy"]).any()
assert len(cash) == len(PERIODS) * len(GROUPS) * 4
display(duplicate_summary)
display(duplicate_by_channel)
display(Markdown(
    f"Платежи вне срока: {duplicate_by_channel['out_of_term_amount'].sum():,.0f} ₽ "
    f"({duplicate_by_channel['out_of_term_amount'].sum() / audit.loc[0, 'revenue'] * 100:.2f}% выручки); "
    f"близкие повторы: {duplicate_by_channel['duplicate_amount'].sum():,.0f} ₽ "
    f"({duplicate_by_channel['duplicate_amount'].sum() / audit.loc[0, 'revenue'] * 100:.3f}%). "
    "Это пересекающиеся признаки: суммы нельзя складывать как непересекающиеся потери."
))


### 1.2. Выручка и темпы роста всех каналов

В первой таблице показаны общая месячная выручка, число успешных платежей,
средний платёж и средняя дневная выручка. Во второй показан рост месячной суммы и средней дневной выручки по каналам
при сравнении марта и июня 2026 года.

График 1 сравнивает рост средней дневной выручки июня относительно марта.
Каналы отсортированы по величине роста; оранжевым выделен канал с минимальным
ростом. Пересчёт на календарный день учитывает разную продолжительность месяцев.

Обе таблицы используют успешные платежи без дополнительных фильтров.

In [ ]:
channel_overview = cash.loc[cash["policy"].eq("Все успешные"),
    ["month", "comparison", "payments", "revenue"]].rename(columns={"comparison": "channel"}).copy()
channel_overview['month'] = pd.to_datetime(channel_overview['month'])
assert not channel_overview.duplicated(['month', 'channel']).any()
monthly_overview = channel_overview.groupby('month')[['revenue', 'payments']].sum()
monthly_overview['Средний платеж, руб.'] = monthly_overview['revenue'] / monthly_overview['payments']
monthly_overview['Выручка в день, руб.'] = monthly_overview['revenue'] / monthly_overview.index.days_in_month
assert np.isclose(monthly_overview['revenue'].sum(), cash.loc[cash['policy'].eq('Все успешные'), 'revenue'].sum(), atol=.01, rtol=0)
channel_revenue = channel_overview.pivot(index='channel', columns='month', values='revenue')
channel_daily = channel_revenue.div(channel_revenue.columns.days_in_month, axis=1)

channel_revenue = channel_revenue.reindex(GROUPS)
channel_daily = channel_daily.reindex(GROUPS)
channel_growth = pd.DataFrame(index=GROUPS)
for start, end in PERIOD_PAIRS:
    label = f"{MONTHS[start.month]} - {MONTHS[end.month]}, %"
    channel_growth[label] = (channel_daily[end] / channel_daily[start] - 1) * 100

display(monthly_overview)
revenue_comparison = channel_growth.copy()
revenue_comparison["Выручка марта, руб."] = channel_revenue[pd.Timestamp("2026-03-01")]
revenue_comparison["Выручка июня, руб."] = channel_revenue[pd.Timestamp("2026-06-01")]
revenue_comparison["Доля выручки июня, %"] = revenue_comparison["Выручка июня, руб."] / revenue_comparison["Выручка июня, руб."].sum() * 100

# Первый столбец channel_growth - рост дневной выручки за март-июнь.
growth = channel_growth.iloc[:, 0].sort_values()

# Выделяем канал с минимальным ростом.
colors = [
    "#F28E2B" if channel == growth.idxmin() else "#9CA3AF"
    for channel in growth.index
]

fig, ax = plt.subplots(figsize=(10, 5))

bars = ax.barh(
    growth.index,
    growth.values,
    color=colors
)

ax.bar_label(bars, fmt="%.2f%%", padding=4)

ax.set_title(f"1. {growth.idxmin()}: рост дневной выручки за март → июнь 2026, {growth.min():.2f}%")
ax.set_xlabel("Июнь относительно марта 2026, % (UTC)")
ax.set_ylabel("Канал")
ax.axvline(0, color="black", linewidth=0.7)
ax.grid(axis="x", alpha=0.2)
ax.set_axisbelow(True)
ax.margins(x=0.15)

plt.tight_layout()
plt.show()

main_growth = channel_growth.iloc[:, 0]
display(Markdown(
    f"При сравнении марта и июня 2026 рост дневной выручки лежит от {main_growth.min():.2f}% "
    f"до {main_growth.max():.2f}%. Минимум - {main_growth.idxmin()}, максимум - "
    f"{main_growth.idxmax()}. Это описательное ранжирование всех каналов; "
    "оно не измеряет эффект источника привлечения."
))
revenue_comparison["Рост месячной суммы, %"] = (
    revenue_comparison["Выручка июня, руб."] / revenue_comparison["Выручка марта, руб."] - 1) * 100
display(revenue_comparison[["Рост месячной суммы, %", "Март - Июнь, %"]])
display(Markdown(
    f"Для {FOCUS} при сравнении марта и июня 2026 месячная сумма изменилась на {revenue_comparison.loc[FOCUS, 'Рост месячной суммы, %']:.2f}%, "
    f"а средняя дневная — на {revenue_comparison.loc[FOCUS, 'Март - Июнь, %']:.2f}%. "
    "Разница связана с длиной месяцев; это два разных показателя."
))


### 1.3. Бизнес-вопрос и устойчивость

**Чем объясняются различия темпов роста средней дневной выручки между каналами
и какое исследование удержания стоит провести первым?**

Проверяем четыре правила: все успешные платежи; только внутри срока;
без близких повторов; внутри срока и без повторов одновременно.
Последний вариант проверяет совместное влияние двух ограничений.
Таблица показывает чувствительность результата, а не доверительный интервал.


In [ ]:
sensitivity_rows = []
for start, end in PERIOD_PAIRS:
    for policy, part in cash.groupby("policy"):
        revenue = part.pivot(index="comparison", columns="month", values="revenue")
        growth = (revenue[end] / end.days_in_month / (revenue[start] / start.days_in_month) - 1) * 100
        for channel in GROUPS:
            sensitivity_rows.append({"Период": f"{MONTHS[start.month]} - {MONTHS[end.month]}",
                "Правило": policy, "Канал": channel, "Рост, %": growth[channel]})
sensitivity = pd.DataFrame(sensitivity_rows)
sensitivity_main = sensitivity[sensitivity["Период"].eq("Март - Июнь")].pivot(
    index="Канал", columns="Правило", values="Рост, %").reindex(GROUPS)
sensitivity_main["В сроке - исходная, п.п."] = sensitivity_main["Только в сроке"] - sensitivity_main["Все успешные"]
sensitivity_main["Без повторов - исходная, п.п."] = sensitivity_main["Без повторов 20 минут"] - sensitivity_main["Все успешные"]
sensitivity_main["Оба фильтра - исходная, п.п."] = (
    sensitivity_main["В сроке и без повторов"] - sensitivity_main["Все успешные"])
display(sensitivity_main)
policy_columns = list(cash["policy"].unique())
rankings = sensitivity_main[policy_columns].idxmin()
display(Markdown("Минимальный рост по правилам: " + "; ".join(
    f"{policy}: {channel}" for policy, channel in rankings.items()) + "."))


## 2. Проверка 12 гипотез

Гипотезы рассматриваются как рабочие объяснения и проверяются с помощью описательных сопоставлений.
Статистическая значимость и причинный эффект здесь не оцениваются.
Для каждой гипотезы указаны метод и условия интерпретации. Результаты
представлены в таблицах и сводке в конце раздела.

**Определения базы:** `active` обозначает подписки, пересекающие месяц; `base_start` обозначает
подписки на начало месяца; `churned` обозначает завершения из этой начальной базы;
`base_end` обозначает базу на конец месяца. Отток = `churned / base_start`.
В начальную базу включаются подписки, удовлетворяющие условиям:
`started_at < начало месяца`, `ended_at >= начало месяца`.
Завершение ровно на границе включается в начальную базу и в отток месяца.
Это отток подписок, а не окончательный уход владельцев.

### Гипотеза 1. Различия роста выручки связаны с ростом активной базы

**Метод:** сравнить рост активной базы всех каналов в двух периодах.
**Ограничение:** совпадение ранжирования не измеряет вклад фактора;
точное разложение следует в разделе 3.

In [ ]:
base = q("""
select m::date as month,
       coalesce(c.acquisition_channel, 'без канала') as comparison,
       count(*) as active,
       count(*) filter (where s.started_at < m) as base_start,
       count(*) filter (where s.started_at >= m) as new_subs,
       count(*) filter (where s.ended_at < m + interval '1 month') as ended_subs,
       count(*) filter (where s.ended_at is null or s.ended_at >= m + interval '1 month') as base_end,
       count(*) filter (where s.started_at < m and s.ended_at < m + interval '1 month') as churned
from generate_series('2026-01-01'::timestamptz, '2026-06-01'::timestamptz, interval '1 month') m
join prime.subscriptions s on s.started_at < m + interval '1 month'
                         and (s.ended_at is null or s.ended_at >= m)
join prime.clients c on c.client_id = s.owner_client_id
group by 1, 2 order by 1, 2
""")
base["month"] = pd.to_datetime(base["month"])
assert not base.duplicated(["month", "comparison"]).any()
assert (base["base_start"] + base["new_subs"] - base["ended_subs"]).eq(base["base_end"]).all()
assert (base["base_start"] + base["new_subs"]).eq(base["active"]).all()
for _, part in base.groupby("comparison"):
    assert np.array_equal(part["base_end"].to_numpy()[:-1], part["base_start"].to_numpy()[1:])


# Общая таблица движения базы используется во всех разрезах.
movement = base.rename(columns={"comparison": "channel", "churned": "churned_start"})
active_comparison = base.pivot(index="month", columns="comparison", values="active")
active_growth = pd.DataFrame({
    f"{MONTHS[a.month]} - {MONTHS[b.month]}, %":
    (active_comparison.loc[b] / active_comparison.loc[a] - 1) * 100
    for a, b in PERIOD_PAIRS
}).reindex(GROUPS)
display(active_growth)


### Гипотеза 2. Повышенный отток ограничивает накопление базы

**Метод:** сравнить новые оформления, отток и чистый прирост относительно
начальной базы каждого канала. Основная сводка охватывает период с марта по июнь, дополнительная охватывает
период с января по июнь. Сводный отток считается как сумма завершений / сумма начальных баз.
Он не равен доле уникальных ушедших подписок за весь период.

График 2 показывает отток каждого месяца. Факторное разложение движения базы
представлено в разделе 3. **Ограничение:** высокая доля оттока не определяет
абсолютную потерю подписок без учёта размера базы.

In [ ]:
for metric in ["new_subs", "ended_subs", "churned"]:
    base[metric + "_pct"] = base[metric] / base["base_start"] * 100
base["net_pct"] = (base["base_end"] - base["base_start"]) / base["base_start"] * 100
assert np.isclose(base["new_subs_pct"] - base["ended_subs_pct"], base["net_pct"]).all()

weighted_churn = base.groupby("comparison")[["base_start", "churned"]].sum()
weighted_churn["Отток, %"] = weighted_churn["churned"] / weighted_churn["base_start"] * 100
base_main = base[base["month"].ge(pd.Timestamp("2026-03-01"))]
flow_main = base_main.groupby("comparison")[["base_start", "new_subs", "churned", "ended_subs"]].sum().reindex(GROUPS)
flow_main["Новые оформления, %"] = flow_main["new_subs"] / flow_main["base_start"] * 100
flow_main["Отток, %"] = flow_main["churned"] / flow_main["base_start"] * 100
flow_main["Средний месячный прирост, %"] = (flow_main["new_subs"] - flow_main["ended_subs"]) / flow_main["base_start"] * 100
display(flow_main)
display(weighted_churn.reindex(GROUPS))

churn_matrix = base.pivot(index="comparison", columns="month", values="churned_pct").reindex(GROUPS)
fig, ax = plt.subplots(figsize=(10, 4))
im = ax.imshow(churn_matrix, aspect="auto", cmap="Blues")
ax.set_yticks(range(len(GROUPS)), GROUPS)
ax.set_xticks(range(6), [MONTHS[m.month] for m in churn_matrix.columns])
for (i, j), value in np.ndenumerate(churn_matrix.to_numpy()):
    ax.text(j, i, f"{value:.2f}", ha="center", va="center",
            color="white" if im.norm(value) > .55 else "black")
top_churn = flow_main["Отток, %"].idxmax()
ax.set(title=f"2. Месячный отток по каналам: январь–июнь 2026", xlabel="Месяц 2026 года (UTC)")
fig.colorbar(im, ax=ax, label="% начальной базы")
plt.tight_layout()
plt.show()


In [ ]:
def churn_by_group(data, group, base_column="base_start"):
    grouped = data.groupby(["channel", group], observed=True)[[base_column, "churned"]].sum()
    grouped["Отток, %"] = grouped["churned"] / grouped[base_column] * 100
    return (grouped["Отток, %"].unstack(group).reindex(GROUPS),
            grouped[base_column].unstack(group).reindex(GROUPS))

def group_evidence(data, group, base_column="base_start", order=None):
    """Числитель и знаменатель рядом с каждой долей, без независимости повторов."""
    table = data.groupby(["channel", group], observed=True)[[base_column, "churned"]].sum()
    table["Отток, %"] = table["churned"] / table[base_column] * 100
    if order is not None:
        table = table.reindex(pd.MultiIndex.from_product([GROUPS, order], names=["channel", group]))
    return table

def describe_groups(data, group, base_column="base_start"):
    """Сопоставить social и остальные внутри одинаковых групп; состава не выравнивает."""
    focus = data.loc[data["channel"].eq(FOCUS)].groupby(group, observed=True)[[base_column, "churned"]].sum()
    others = data.loc[~data["channel"].eq(FOCUS)].groupby(group, observed=True)[[base_column, "churned"]].sum()
    joined = focus.join(others, lsuffix="_social", rsuffix="_others", how="inner")
    joined = joined[(joined[base_column + "_social"] > 0) & (joined[base_column + "_others"] > 0)]
    joined["social, %"] = joined["churned_social"] / joined[base_column + "_social"] * 100
    joined["Остальные, %"] = joined["churned_others"] / joined[base_column + "_others"] * 100
    joined["Разрыв, п.п."] = joined["social, %"] - joined["Остальные, %"]
    if joined.empty:
        return joined, "Нет общих групп с ненулевыми знаменателями; сравнение невозможно."
    gap = joined["Разрыв, п.п."]
    result = (f"В {int(gap.gt(0).sum())} из {len(gap)} общих групп отток social выше остальных; "
              f"разрыв от {gap.min():+.2f} до {gap.max():+.2f} п.п. "
              "Этот разрез не устраняет наблюдаемое различие полностью; вклад состава не оценён."
              if gap.gt(0).all() else
              f"Направление различий неоднородно: social выше остальных в {int(gap.gt(0).sum())} "
              f"из {len(gap)} групп; разрыв {gap.min():+.2f}…{gap.max():+.2f} п.п. "
              "Нельзя объявить признак объяснением общего разрыва без выравнивания состава.")
    return joined, result

evidence_tables = {}
group_findings = {}
group_comparisons = {}


### Гипотеза 3. Возраст подписки связан с различиями оттока

**Метод:** сравнить отток всех каналов внутри пяти групп возраста подписки на
начало месяца, январь-июнь. Для каждого процента показаны число завершений и сумма начальных баз.
Возрастные группы расположены от младших к старшим; полные таблицы экспортируются.
**Ограничение:** отдельный разрез не измеряет вклад состава и не устраняет
совместные различия клиентов.


In [ ]:
age_churn = q("""
    with base as (
        select
            m as month_start,
            coalesce(c.acquisition_channel, 'без канала') as channel,
            s.ended_at,
            floor(
                extract(epoch from (m - s.started_at)) / 86400
            ) as age_days

        from generate_series(
            '2026-01-01'::timestamptz,
            '2026-06-01'::timestamptz,
            interval '1 month'
        ) m

        join prime.subscriptions s
          on s.started_at < m
         and (s.ended_at is null or s.ended_at >= m)

        join prime.clients c
          on c.client_id = s.owner_client_id
    ),

    grouped as (
        select *,
            case
                when age_days < 30 then 'Менее 30 дней'
                when age_days < 90 then '30-89 дней'
                when age_days < 180 then '90-179 дней'
                when age_days < 365 then '180-364 дня'
                else '365 дней и более'
            end as age_group
        from base
    )

    select
        month_start::date as month,
        channel,
        age_group,
        count(*) as base_start,
        count(*) filter (
            where ended_at < month_start + interval '1 month'
        ) as churned

    from grouped
    group by 1, 2, 3
    order by 1, 2, 3
""")

age_churn["month"] = pd.to_datetime(age_churn["month"])

AGE_ORDER = [
    "Менее 30 дней",
    "30-89 дней",
    "90-179 дней",
    "180-364 дня",
    "365 дней и более"
]

assert not age_churn.duplicated(
    ["month", "channel", "age_group"]
).any()

assert age_churn["churned"].le(age_churn["base_start"]).all()

check_base_totals(age_churn)

age_rates, age_counts = churn_by_group(age_churn, "age_group")
age_rates = age_rates.reindex(columns=AGE_ORDER)
age_counts = age_counts.reindex(columns=AGE_ORDER)
display(age_rates)

evidence_tables["hypothesis_3"] = group_evidence(age_churn, "age_group", "base_start", AGE_ORDER)
display(evidence_tables["hypothesis_3"])
group_comparisons["3"], group_findings["3"] = describe_groups(age_churn, "age_group", "base_start")
display(Markdown("**Результат.** " + group_findings["3"]))


### Гипотеза 4. Возраст и стаж владельца связаны с различиями оттока

**Метод:** отдельно сравнить каналы внутри возрастных групп владельца и групп
стажа регистрации, январь-июнь. Возраст оценивается по году рождения;
признаки определяются до наблюдения оттока.
**Ограничение:** два отдельных разреза не равны одновременной корректировке.


In [ ]:
client_profile = q("""
    with base as (
        select
            m as month_start,
            s.owner_client_id,
            s.ended_at,
            coalesce(c.acquisition_channel, 'без канала') as channel,
            extract(year from m)::int - c.birth_year as client_age,
            floor(
                extract(epoch from (m - c.registered_at)) / 86400
            ) as tenure_days

        from generate_series(
            '2026-01-01'::timestamptz,
            '2026-06-01'::timestamptz,
            interval '1 month'
        ) m

        join prime.subscriptions s
          on s.started_at < m
         and (s.ended_at is null or s.ended_at >= m)

        join prime.clients c
          on c.client_id = s.owner_client_id
    ),

    classified as (
        select *,
            case
                when client_age is null
                  or client_age < 0
                  or client_age > 110 then 'Нет корректных данных'
                when client_age < 25 then 'До 25 лет'
                when client_age < 35 then '25-34 года'
                when client_age < 45 then '35-44 года'
                when client_age < 55 then '45-54 года'
                else '55 лет и старше'
            end as client_age_group,

            case
                when tenure_days is null
                  or tenure_days < 0 then 'Нет корректных данных'
                when tenure_days < 90 then 'Менее 90 дней'
                when tenure_days < 365 then '90-364 дня'
                else '365 дней и более'
            end as tenure_group
        from base
    )

    select month_start::date as month, channel,
           'Возраст клиента' as dimension, client_age_group as client_group,
           count(distinct owner_client_id) as owners, count(*) as base_start,
           count(*) filter (where ended_at < month_start + interval '1 month') as churned
    from classified group by 1, 2, 3, 4
    union all
    select month_start::date as month, channel,
           'Стаж клиента' as dimension, tenure_group as client_group,
           count(distinct owner_client_id) as owners, count(*) as base_start,
           count(*) filter (where ended_at < month_start + interval '1 month') as churned
    from classified group by 1, 2, 3, 4
    order by 1, 2, 3, 4
""")

client_profile["month"] = pd.to_datetime(client_profile["month"])

assert not client_profile.duplicated(
    ["month", "channel", "dimension", "client_group"]
).any()

assert client_profile["churned"].le(
    client_profile["base_start"]
).all()

assert client_profile["owners"].le(
    client_profile["base_start"]
).all()

for _, part in client_profile.groupby("dimension"):
    check_base_totals(part)

profile_rates, profile_counts = {}, {}
for dimension, part in client_profile.groupby("dimension"):
    profile_rates[dimension], profile_counts[dimension] = churn_by_group(part, "client_group")
    display(Markdown(f"**{dimension}: отток, %.**"))
    display(profile_rates[dimension])

profile_findings = []
for number, (dimension, part) in enumerate(client_profile.groupby("dimension"), start=1):
    evidence_tables[f"hypothesis_4_{number}"] = group_evidence(part, "client_group")
    display(evidence_tables[f"hypothesis_4_{number}"])
    comparison, finding = describe_groups(part, "client_group")
    group_comparisons[f"4_{number}"] = comparison
    profile_findings.append(dimension + ": " + finding)
group_findings["4"] = " ".join(profile_findings)
display(Markdown("**Результат.** " + group_findings["4"]))


### Гипотеза 5. Предшествующая активность связана с различиями оттока

**Метод:** число дней использования сервисов владельцем за 30 дней до месяца
и последующий отток. Включаются подписки возрастом не менее 30 дней и месяцы
без обнаруженных разрывов предшествующего окна; список месяцев выводится.
**Ограничение:** использование участниками не измерено, высокая активность
может иметь малую базу, причинное влияние активности не установлено.


In [ ]:
activity_churn = q("""
    with months as (
        select m as month_start
        from generate_series(
            '2026-01-01'::timestamptz,
            '2026-06-01'::timestamptz,
            interval '1 month'
        ) m
    ),

    active_subs as (
        select
            m.month_start,
            s.subscription_id,
            s.owner_client_id,
            s.started_at,
            s.ended_at,
            coalesce(c.acquisition_channel, 'без канала') as channel

        from months m

        join prime.subscriptions s
          on s.started_at < m.month_start
         and (
             s.ended_at is null
             or s.ended_at >= m.month_start
         )

        join prime.clients c
          on c.client_id = s.owner_client_id
    ),

    owners as (
        select distinct month_start, owner_client_id
        from active_subs
        where started_at <= month_start - interval '30 days'
    ),

    usage_by_owner as (
        select
            o.month_start,
            o.owner_client_id,
            count(distinct u.started_at::date) as active_days

        from owners o

        left join prime.service_usage u
          on u.client_id = o.owner_client_id
         and u.started_at >= o.month_start - interval '30 days'
         and u.started_at < o.month_start
         and u.ended_at > u.started_at
         and u.ended_at <= o.month_start

        group by 1, 2
    ),

    classified as (
        select
            s.*,
            case
                when s.started_at > s.month_start - interval '30 days'
                    then 'Подписка младше 30 дней'
                when coalesce(u.active_days, 0) = 0
                    then '0 дней'
                when u.active_days <= 3
                    then '1-3 дня'
                when u.active_days <= 10
                    then '4-10 дней'
                else '11-30 дней'
            end as activity_group

        from active_subs s

        left join usage_by_owner u
          on u.month_start = s.month_start
         and u.owner_client_id = s.owner_client_id
    )

    select
        month_start::date as month,
        channel,
        activity_group,
        count(*) as base_start,
        count(*) filter (
            where ended_at < month_start + interval '1 month'
        ) as churned

    from classified
    group by 1, 2, 3
    order by 1, 2, 3
""")

activity_churn["month"] = pd.to_datetime(
    activity_churn["month"]
)

assert not activity_churn.duplicated(
    ["month", "channel", "activity_group"]
).any()

assert activity_churn["churned"].le(
    activity_churn["base_start"]
).all()

check_base_totals(activity_churn)

# Проверяем наличие записей использования по дням.
usage_coverage = q("""
    select started_at::date as day, count(*) as sessions
    from prime.service_usage
    where started_at >= '2026-01-01'::timestamptz - interval '30 days'
      and started_at < '2026-06-01'::timestamptz
    group by 1
    order by 1
""")

usage_coverage["day"] = pd.to_datetime(usage_coverage["day"])
available_days = pd.DatetimeIndex(usage_coverage["day"])

coverage_rows = []

for month in PERIODS:
    window = pd.date_range(
        month - pd.Timedelta(days=30),
        periods=30,
        freq="D"
    )

    coverage_rows.append({
        "month": month,
        "missing_days": len(window.difference(available_days))
    })

activity_coverage = pd.DataFrame(coverage_rows)

# Окна с явными пропусками пока не используем.
valid_months = activity_coverage.loc[
    activity_coverage["missing_days"].eq(0),
    "month"
]

assert len(valid_months) > 0, "Нет окон без явных пропусков"

ACTIVITY_ORDER = [
    "0 дней",
    "1-3 дня",
    "4-10 дней",
    "11-30 дней"
]

activity_sample = activity_churn[
    activity_churn["month"].isin(valid_months)
    & activity_churn["activity_group"].isin(ACTIVITY_ORDER)
].copy()

activity_sample["comparison"] = activity_sample["channel"].copy()

activity_rates, activity_counts = churn_by_group(activity_sample, "activity_group")
display(activity_coverage)
display(activity_rates)

evidence_tables["hypothesis_5"] = group_evidence(activity_sample, "activity_group", "base_start", ACTIVITY_ORDER)
display(evidence_tables["hypothesis_5"])
group_comparisons["5"], group_findings["5"] = describe_groups(activity_sample, "activity_group", "base_start")
display(Markdown("**Результат.** " + group_findings["5"]))


### Гипотеза 6. Зарегистрированные причины отмен различаются по каналам

**Метод:** сравнить доли причин среди завершившихся подписок и вклад каждой
причины в отток начальной базы, январь-июнь. Сумма вкладов должна давать общий
отток. **Ограничение:** частая записанная причина не равна установленной
первопричине; одинаковая структура причин может сочетаться с разным оттоком.

In [ ]:
cancel_reasons = q("""
    with base as (
        select
            m as month_start,
            coalesce(c.acquisition_channel, 'без канала') as channel,
            s.ended_at,
            coalesce(
                nullif(trim(s.cancel_reason), ''),
                'Причина не указана'
            ) as reason

        from generate_series(
            '2026-01-01'::timestamptz,
            '2026-06-01'::timestamptz,
            interval '1 month'
        ) m

        join prime.subscriptions s
          on s.started_at < m
         and (s.ended_at is null or s.ended_at >= m)

        join prime.clients c
          on c.client_id = s.owner_client_id
    )

    select
        month_start::date as month,
        channel,
        reason,
        count(*) as churned

    from base
    where ended_at < month_start + interval '1 month'
    group by 1, 2, 3
    order by 1, 2, 3
""")

cancel_reasons["month"] = pd.to_datetime(
    cancel_reasons["month"]
)

assert not cancel_reasons.duplicated(
    ["month", "channel", "reason"]
).any()

# Сверяем число завершений с предыдущим расчетом.
reason_check = (
    cancel_reasons.groupby(["month", "channel"])["churned"]
    .sum()
)

expected = (
    movement.set_index(["month", "channel"])["churned_start"]
    .sort_index()
)

reason_check = reason_check.reindex(
    expected.index,
    fill_value=0
)

assert np.array_equal(reason_check, expected)

cancel_reasons["comparison"] = cancel_reasons["channel"].copy()

# Начальную базу считаем отдельно от причин.
reason_base = movement.copy()

reason_base["comparison"] = reason_base["channel"].copy()

base_totals = (
    reason_base.groupby("comparison")["base_start"]
    .sum()
)

reason_summary = (
    cancel_reasons.groupby(
        ["reason", "comparison"],
        as_index=False
    )["churned"]
    .sum()
)

# Сохраняем причины, которые есть только в одной группе.
all_reasons = sorted(reason_summary["reason"].unique())

reason_summary = (
    reason_summary.set_index(["reason", "comparison"])
    .reindex(
        pd.MultiIndex.from_product(
            [all_reasons, GROUPS],
            names=["reason", "comparison"]
        ),
        fill_value=0
    )
    .reset_index()
)

reason_summary["base_start"] = (
    reason_summary["comparison"].map(base_totals)
)

reason_summary["share_pct"] = (
    reason_summary["churned"]
    / reason_summary.groupby(
        "comparison"
    )["churned"].transform("sum")
    * 100
)

reason_summary["churn_contribution_pct"] = (
    reason_summary["churned"]
    / reason_summary["base_start"]
    * 100
)


reason_structure = reason_summary.pivot(index="comparison", columns="reason", values="share_pct").reindex(GROUPS)
reason_rates = reason_summary.pivot(index="comparison", columns="reason", values="churn_contribution_pct").reindex(GROUPS)
assert np.isclose(reason_rates.sum(axis=1), weighted_churn["Отток, %"].reindex(GROUPS)).all()
display(reason_structure)
display(reason_rates)
focus_reasons = reason_summary[reason_summary["comparison"].eq(FOCUS)].sort_values("share_pct", ascending=False)
reason_result = "Причины отмен social за январь–июнь 2026: " + "; ".join(
    f"{row.reason}: {row.share_pct:.2f}% отмен, {row.churn_contribution_pct:.2f} п.п. оттока"
    for row in focus_reasons.head(2).itertuples())
display(Markdown("**Результат для social.** " + reason_result +
    ". Это записанные причины; их доли не доказывают причинный эффект цены или использования."))


### Гипотеза 7. Совместное использование связано с различиями оттока

**Метод:** сравнить отток каналов при отсутствии участников, одном и двух и более
участниках, январь-июнь. Владелец исключён из числа участников; состав определяется
на начало месяца. **Ограничение:** связь не доказывает эффект подключения
участника, полнота истории изменений участия отдельно не подтверждена.


In [ ]:
sharing_churn = q("""
    with base as (
        select
            m as month_start,
            s.subscription_id,
            s.owner_client_id,
            s.ended_at,
            coalesce(c.acquisition_channel, 'без канала') as channel

        from generate_series(
            '2026-01-01'::timestamptz,
            '2026-06-01'::timestamptz,
            interval '1 month'
        ) m

        join prime.subscriptions s
          on s.started_at < m
         and (s.ended_at is null or s.ended_at >= m)

        join prime.clients c
          on c.client_id = s.owner_client_id
    ),

    member_counts as (
        select
            b.month_start,
            b.subscription_id,
            b.channel,
            b.ended_at,
            count(distinct sm.client_id) as n_members

        from base b

        left join prime.subscription_members sm
          on sm.subscription_id = b.subscription_id
         and sm.client_id <> b.owner_client_id
         and sm.joined_at <= b.month_start
         and (
             sm.left_at is null
             or sm.left_at >= b.month_start
         )

        group by
            b.month_start,
            b.subscription_id,
            b.channel,
            b.ended_at
    ),

    classified as (
        select *,
            case
                when n_members = 0 then 'Без участников'
                when n_members = 1 then 'Один участник'
                else 'Два и более участников'
            end as sharing_group
        from member_counts
    )

    select
        month_start::date as month,
        channel,
        sharing_group,
        count(*) as base_start,
        count(*) filter (
            where ended_at < month_start + interval '1 month'
        ) as churned

    from classified
    group by 1, 2, 3
    order by 1, 2, 3
""")

sharing_churn["month"] = pd.to_datetime(
    sharing_churn["month"]
)

assert not sharing_churn.duplicated(
    ["month", "channel", "sharing_group"]
).any()

assert sharing_churn["churned"].le(
    sharing_churn["base_start"]
).all()

check_base_totals(sharing_churn)

sharing_rates, sharing_counts = churn_by_group(sharing_churn, "sharing_group")
display(sharing_rates)

evidence_tables["hypothesis_7"] = group_evidence(sharing_churn, "sharing_group", "base_start", None)
display(evidence_tables["hypothesis_7"])
group_comparisons["7"], group_findings["7"] = describe_groups(sharing_churn, "sharing_group", "base_start")
display(Markdown("**Результат.** " + group_findings["7"]))


### Гипотеза 8. Региональный состав связан с различиями оттока

**Метод:** сравнить все каналы внутри одинаковых регионов, январь-июнь.
Берём регионы с суммой начальных баз не менее 100 у каждого канала и показываем
десять крупнейших по совокупной базе. Охват указывает, какая доля наблюдений
попала в общие регионы. Порог 100 не является тестом значимости.
**Ограничение:** сравнение внутри регионов не измеряет вклад регионального состава.


In [ ]:
region_churn = q("""
    select
        m::date as month,
        coalesce(c.acquisition_channel, 'без канала') as channel,
        coalesce(c.region_code::text, 'Не указан') as region,
        count(*) as base_start,
        count(*) filter (
            where s.ended_at < m + interval '1 month'
        ) as churned

    from generate_series(
        '2026-01-01'::timestamptz,
        '2026-06-01'::timestamptz,
        interval '1 month'
    ) m

    join prime.subscriptions s
      on s.started_at < m
     and (s.ended_at is null or s.ended_at >= m)

    join prime.clients c
      on c.client_id = s.owner_client_id

    group by 1, 2, 3
    order by 1, 2, 3
""")

region_churn["month"] = pd.to_datetime(
    region_churn["month"]
)

assert not region_churn.duplicated(
    ["month", "channel", "region"]
).any()

assert region_churn["churned"].le(
    region_churn["base_start"]
).all()

check_base_totals(region_churn)

region_rates, region_counts = churn_by_group(region_churn, "region")
common_regions = region_counts.ge(100).all(axis=0)
region_common_rates = region_rates.loc[:, common_regions]
top_regions = region_counts.loc[:, common_regions].sum(axis=0).nlargest(10).index
display(region_common_rates[top_regions])
region_coverage = pd.DataFrame({
    "Общих регионов": int(common_regions.sum()),
    "Охват базы, %": region_counts.loc[:, common_regions].sum(axis=1) / region_counts.sum(axis=1) * 100
})
display(region_coverage)

evidence_tables["hypothesis_8"] = group_evidence(region_churn[region_churn["region"].isin(top_regions)], "region", "base_start", None)
display(evidence_tables["hypothesis_8"])
group_comparisons["8"], group_findings["8"] = describe_groups(region_churn[region_churn["region"].isin(region_common_rates.columns)], "region", "base_start")
display(Markdown("**Результат.** " + group_findings["8"]))

evidence_tables["hypothesis_8_all_regions"] = group_evidence(region_churn, "region")
region_coverage["Охват показанных 10 регионов, %"] = (
    region_counts[top_regions].sum(axis=1) / region_counts.sum(axis=1) * 100)
display(region_coverage)


### Гипотеза 9. Используемые сервисы связаны с различиями оттока

**Метод:** сравнить отток всех каналов среди владельцев, использовавших один и
тот же сервис за предыдущие 30 дней. Ограничения возраста и календаря такие же,
как в гипотезе 5. Показываем отток по сервисам и доли использовавших их владельцев
в соответствующей базе подписок. **Ограничение:** группы сервисов пересекаются,
их показатели нельзя складывать; разрез не доказывает влияние сервиса.


In [ ]:
service_churn = q("""
    with base as (
        select
            m as month_start,
            s.subscription_id,
            s.owner_client_id,
            s.ended_at,
            coalesce(c.acquisition_channel, 'без канала') as channel

        from generate_series(
            '2026-01-01'::timestamptz,
            '2026-06-01'::timestamptz,
            interval '1 month'
        ) m

        join prime.subscriptions s
          on s.started_at <= m - interval '30 days'
         and (s.ended_at is null or s.ended_at >= m)

        join prime.clients c
          on c.client_id = s.owner_client_id
    ),

    owners as (
        select distinct month_start, owner_client_id
        from base
    ),

    used_services as (
        select distinct
            o.month_start,
            o.owner_client_id,
            u.service_id

        from owners o

        join prime.service_usage u
          on u.client_id = o.owner_client_id
         and u.started_at >= o.month_start - interval '30 days'
         and u.started_at < o.month_start
         and u.ended_at > u.started_at
         and u.ended_at <= o.month_start
    )

    select
        b.month_start::date as month,
        b.channel,
        u.service_id,
        sv.service_name,
        count(*) as service_base,
        count(*) filter (
            where b.ended_at < b.month_start + interval '1 month'
        ) as churned

    from base b

    join used_services u
      on u.month_start = b.month_start
     and u.owner_client_id = b.owner_client_id

    join prime.services sv
      on sv.service_id = u.service_id

    group by 1, 2, 3, 4
    order by 1, 2, 3
""")

service_churn["month"] = pd.to_datetime(
    service_churn["month"]
)

assert not service_churn.duplicated(
    ["month", "channel", "service_id"]
).any()

assert service_churn["churned"].le(
    service_churn["service_base"]
).all()

# Оставляем те же месяцы, что в анализе активности.
service_churn = service_churn[
    service_churn["month"].isin(valid_months)
].copy()

# Начальная база включает и владельцев без использования.
eligible_base = (
    activity_churn[
        activity_churn["month"].isin(valid_months)
        & activity_churn["activity_group"].isin(ACTIVITY_ORDER)
    ]
    .groupby(["month", "channel"], as_index=False)["base_start"]
    .sum()
)

service_check = service_churn.merge(
    eligible_base,
    on=["month", "channel"],
    how="left",
    validate="m:1"
)

assert service_check["base_start"].notna().all()

assert service_check["service_base"].le(
    service_check["base_start"]
).all()

base_by_channel = (
    eligible_base.groupby("channel")["base_start"]
    .sum()
)

service_summary = (
    service_churn.groupby(
        ["service_id", "service_name", "channel"],
        as_index=False
    )[["service_base", "churned"]]
    .sum()
)

service_summary["base_start"] = (
    service_summary["channel"].map(base_by_channel)
)

service_summary["usage_share_pct"] = (
    service_summary["service_base"]
    / service_summary["base_start"]
    * 100
)

service_summary["churn_pct"] = (
    service_summary["churned"]
    / service_summary["service_base"]
    * 100
)




service_rates, service_counts = churn_by_group(service_churn, "service_name", "service_base")
display(service_rates)
display(service_summary.pivot(index="channel", columns="service_name", values="usage_share_pct").reindex(GROUPS))
evidence_tables["hypothesis_9"] = group_evidence(service_churn, "service_name", "service_base", None)
display(evidence_tables["hypothesis_9"])
group_comparisons["9"], group_findings["9"] = describe_groups(service_churn, "service_name", "service_base")
display(Markdown("**Результат.** " + group_findings["9"]))


### Гипотеза 10. Нерешённые обращения связаны с повышенным оттоком

**Метод:** обращения владельца за 30 дней до месяца и последующий отток;
состояние решения фиксируется на начало месяца. Молодые подписки, некорректные
даты и неполные предшествующие окна исключаются.
Сопоставляем отток, размер группы и её долю по всем каналам.
Условный избыток = доля группы * разница оттока с группой без обращений.
**Ограничение:** это оценка масштаба наблюдаемой связи, а не эффект улучшения поддержки.

In [ ]:
support_churn = q("""
    with base as (
        select
            m as month_start,
            s.subscription_id,
            s.owner_client_id,
            s.started_at,
            s.ended_at,
            coalesce(c.acquisition_channel, 'без канала') as channel

        from generate_series(
            '2026-01-01'::timestamptz,
            '2026-06-01'::timestamptz,
            interval '1 month'
        ) m

        join prime.subscriptions s
          on s.started_at < m
         and (s.ended_at is null or s.ended_at >= m)

        join prime.clients c
          on c.client_id = s.owner_client_id
    ),

    owners as (
        select distinct month_start, owner_client_id
        from base
        where started_at <= month_start - interval '30 days'
    ),

    tickets as (
        select
            o.month_start,
            o.owner_client_id,
            count(t.ticket_id) as n_tickets,

            count(*) filter (
                where t.ticket_id is not null
                  and (
                      t.resolved_at is null
                      or t.resolved_at >= o.month_start
                  )
            ) as open_tickets,

            count(*) filter (
                where t.resolved_at < t.created_at
            ) as bad_dates

        from owners o

        left join prime.support_tickets t
          on t.client_id = o.owner_client_id
         and t.created_at >= o.month_start - interval '30 days'
         and t.created_at < o.month_start

        group by 1, 2
    ),

    classified as (
        select
            b.*,
            case
                when b.started_at > b.month_start - interval '30 days'
                    then 'Подписка младше 30 дней'
                when coalesce(t.bad_dates, 0) > 0
                    then 'Некорректные даты'
                when coalesce(t.n_tickets, 0) = 0
                    then 'Обращений не было'
                when t.open_tickets > 0
                    then 'Есть нерешенные обращения'
                else 'Все обращения решены'
            end as support_group

        from base b

        left join tickets t
          on t.month_start = b.month_start
         and t.owner_client_id = b.owner_client_id
    )

    select
        month_start::date as month,
        channel,
        support_group,
        count(*) as base_start,
        count(*) filter (
            where ended_at < month_start + interval '1 month'
        ) as churned

    from classified
    group by 1, 2, 3
    order by 1, 2, 3
""")

support_churn["month"] = pd.to_datetime(
    support_churn["month"]
)

assert not support_churn.duplicated(
    ["month", "channel", "support_group"]
).any()

assert support_churn["churned"].le(
    support_churn["base_start"]
).all()

check_base_totals(support_churn)

# Проверяем явные разрывы в записях поддержки.
support_dates = q("""
    select distinct created_at::date as day
    from prime.support_tickets
    where created_at >=
        '2026-01-01'::timestamptz - interval '30 days'
      and created_at < '2026-06-01'::timestamptz
""")

available_dates = pd.DatetimeIndex(
    pd.to_datetime(support_dates["day"])
)

coverage_rows = []

for month in PERIODS:
    window = pd.date_range(
        month - pd.Timedelta(days=30),
        periods=30,
        freq="D"
    )

    coverage_rows.append({
        "month": month,
        "missing_days": len(window.difference(available_dates))
    })

support_coverage = pd.DataFrame(coverage_rows)

valid_support_months = support_coverage.loc[
    support_coverage["missing_days"].eq(0),
    "month"
]

assert len(valid_support_months) > 0

SUPPORT_ORDER = [
    "Обращений не было",
    "Все обращения решены",
    "Есть нерешенные обращения"
]


support_sample = support_churn[
    support_churn["month"].isin(valid_support_months)
    & support_churn["support_group"].isin(SUPPORT_ORDER)
].copy()

support_sample["comparison"] = support_sample["channel"].copy()

support_summary = (
    support_sample.groupby(
        ["support_group", "comparison"],
        as_index=False
    )[["base_start", "churned"]]
    .sum()
)

support_summary["share_pct"] = (
    support_summary["base_start"]
    / support_summary.groupby(
        "comparison"
    )["base_start"].transform("sum")
    * 100
)

support_summary["churn_pct"] = (
    support_summary["churned"]
    / support_summary["base_start"]
    * 100
)



support_rates, support_counts = churn_by_group(support_sample, "support_group")
support_rates = support_rates.reindex(columns=SUPPORT_ORDER)
support_counts = support_counts.reindex(columns=SUPPORT_ORDER)
support_impact = pd.DataFrame(index=GROUPS)
support_impact["Наблюдений с нерешёнными"] = support_counts["Есть нерешенные обращения"]
support_impact["Доля группы, %"] = support_counts["Есть нерешенные обращения"] / support_counts.sum(axis=1) * 100
support_impact["Разница оттока, п.п."] = support_rates["Есть нерешенные обращения"] - support_rates["Обращений не было"]
support_impact["Условный избыток оттока, п.п."] = support_impact["Доля группы, %"] / 100 * support_impact["Разница оттока, п.п."]
display(support_coverage)
display(support_rates)
display(support_impact)

evidence_tables["hypothesis_10"] = group_evidence(support_sample, "support_group", "base_start", SUPPORT_ORDER)
display(evidence_tables["hypothesis_10"])
group_comparisons["10"], group_findings["10"] = describe_groups(support_sample, "support_group", "base_start")
display(Markdown("**Результат.** " + group_findings["10"]))

support_focus = support_impact.loc[FOCUS]
support_result = (
    f"У social нерешённые обращения: {support_focus['Наблюдений с нерешёнными']:.0f} наблюдений, "
    f"{support_focus['Доля группы, %']:.3f}% базы; разница оттока "
    f"{support_focus['Разница оттока, п.п.']:+.2f} п.п. Условный избыток "
    f"{support_focus['Условный избыток оттока, п.п.']:+.4f} п.п. "
    "Масштаб этой связи нужно сопоставлять с общим разрывом; это не эффект решения обращений."
)
display(Markdown("**Результат.** " + support_result))


### Гипотеза 11. Ошибки оплаты связаны с последующим оттоком

**Метод:** сначала сравнить долю `failed` среди попыток оплаты в полных месяцах.
Затем выбрать первую допустимую попытку `failed` или `success` каждой подписки
в январе-июне и сравнить завершения за следующие 30 суток по всем каналам.
Допустимая попытка находится внутри `[started_at, ended_at)`.
При одинаковом времени порядок задаёт `payment_id`.
Выбираем первое событие до исключения неполных окон, без замены более поздним.

В таблице рядом с долями показываем число подписок в обеих группах.
**Ограничение:** различия сырые, состав групп по возрасту, тарифу и месяцу
не выровнен. Это первое событие периода, не всей истории подписки.
Результат не измеряет причинное влияние ошибки и не доказывает эффект её устранения.

In [ ]:
# Для доли ошибок оставляем месяцы без обнаруженных дней без записей платежей.
complete_months = []
for month in PERIODS:
    end = month + pd.offsets.MonthBegin(1)
    if not ((missing_days >= month) & (missing_days < end)).any():
        complete_months.append(month)

payment_frequency = q("""
select date_trunc('month', p.paid_at)::date as month,
       coalesce(c.acquisition_channel, 'без канала') as channel,
       count(*) as attempts,
       count(*) filter (where p.status = 'failed') as failed
from prime.payments p
join prime.subscriptions s on s.subscription_id = p.subscription_id
join prime.clients c on c.client_id = s.owner_client_id
where p.status in ('success', 'failed')
  and p.paid_at >= '2026-01-01'::timestamptz
  and p.paid_at < '2026-07-01'::timestamptz
group by 1, 2
""")
payment_frequency["month"] = pd.to_datetime(payment_frequency["month"])
payment_frequency = payment_frequency[payment_frequency["month"].isin(complete_months)]
payment_frequency = payment_frequency.groupby("channel")[["attempts", "failed"]].sum().reindex(GROUPS)
payment_frequency["Доля failed, %"] = payment_frequency["failed"] / payment_frequency["attempts"] * 100
display(payment_frequency)

# MIN находит дату; сортировка и drop_duplicates разрешают совпадения времени.
payment_cohort = q("""
with eligible as (
    select p.payment_id, p.subscription_id, p.paid_at, p.status, s.ended_at,
           coalesce(c.acquisition_channel, 'без канала') as channel
    from prime.payments p
    join prime.subscriptions s on s.subscription_id = p.subscription_id
    join prime.clients c on c.client_id = s.owner_client_id
    where p.status in ('success', 'failed')
      and p.paid_at >= '2026-01-01'::timestamptz
      and p.paid_at < '2026-07-01'::timestamptz
      and p.paid_at >= s.started_at
      and (s.ended_at is null or p.paid_at < s.ended_at)
), first_dates as (
    select subscription_id, min(paid_at) as paid_at
    from eligible group by 1
)
select e.* from eligible e
join first_dates f on f.subscription_id = e.subscription_id and f.paid_at = e.paid_at
""")
payment_cohort = payment_cohort.sort_values(["paid_at", "payment_id"]).drop_duplicates("subscription_id")
assert payment_cohort["subscription_id"].is_unique
for column in ["paid_at", "ended_at"]:
    payment_cohort[column] = pd.to_datetime(payment_cohort[column], utc=True)
payment_cohort["window_end"] = payment_cohort["paid_at"] + pd.Timedelta(days=30)
last_payment = q("select max(paid_at) as last_at from prime.payments").loc[0, "last_at"]
last_payment = pd.to_datetime(last_payment, utc=True)
assert pd.notna(last_payment), "Нет истории платежей"
complete = payment_cohort["window_end"].le(last_payment)
for day in missing_days.tz_localize("UTC"):
    intersects = payment_cohort["paid_at"].lt(day + pd.Timedelta(days=1)) & payment_cohort["window_end"].ge(day)
    complete &= ~intersects
payment_cohort_coverage = pd.DataFrame({"До ограничения": payment_cohort.groupby("channel").size(),
    "Полных окон": payment_cohort.loc[complete].groupby("channel").size()}).reindex(GROUPS).fillna(0)
payment_cohort_coverage["Охват, %"] = payment_cohort_coverage["Полных окон"] / payment_cohort_coverage["До ограничения"].replace(0, np.nan) * 100
payment_cohort = payment_cohort.loc[complete].copy()
payment_cohort["ended_30"] = payment_cohort["ended_at"].notna() & payment_cohort["ended_at"].le(payment_cohort["window_end"])
payment_counts = payment_cohort.groupby(["channel", "status"]).agg(
    subscriptions=("subscription_id", "size"), ended=("ended_30", "sum"))
status_counts = payment_counts["subscriptions"].unstack("status").reindex(index=GROUPS, columns=["failed", "success"])
status_rates = (payment_counts["ended"] / payment_counts["subscriptions"] * 100).unstack("status").reindex(index=GROUPS, columns=["failed", "success"])
payment_comparison = pd.DataFrame({"Подписок failed": status_counts["failed"],
    "Подписок success": status_counts["success"], "Завершились после failed, %": status_rates["failed"],
    "Завершились после success, %": status_rates["success"]})
payment_comparison["Разница failed - success, п.п."] = status_rates["failed"] - status_rates["success"]
display(payment_cohort_coverage)
display(payment_comparison)

focus_payment = payment_comparison.loc[FOCUS]
payment_difference = focus_payment["Разница failed - success, п.п."]
payment_result = (
    f"Для social завершились за 30 суток: после failed "
    f"{focus_payment['Завершились после failed, %']:.2f}%, после success "
    f"{focus_payment['Завершились после success, %']:.2f}%; разница {payment_difference:+.2f} п.п. "
    + ("Направление соответствует предположению о повышенном оттоке после failed. "
       if payment_difference > 0 else "Предположение о более высоком оттоке после failed описательно не поддержано. ")
    + "Группы по возрасту, тарифу и месяцу не выровнены; эффект ошибки не установлен."
)
display(Markdown("**Результат.** " + payment_result))


### Гипотеза 12. Основное отставание social возникает на первом продлении

Сравниваем переходы 1→2, 2→3, 3→4, 4→5 и 5→6 успешных оплат. В знаменателе учитываются
подписки с предыдущей оплатой в январе–июне и полным 35-дневным последующим окном;
в числителе учитываются подписки со следующей оплатой за 31 сутки. Порог 35 суток используется
для чувствительности на той же выборке. Учитываются платежи внутри срока подписки,
повторы одной суммы менее чем через 20 минут убираются до нумерации.

**Полнота истории до события:** учитываются подписки, начавшиеся с февраля 2025.
Дополнительно исключаются индексные события, если между началом подписки и событием
есть хотя бы один календарный день без платежей во всей базе. Это защищает номер
оплаты от известного апрельского разрыва, а не только последующее окно. Наличие
событий за день не доказывает полной загрузки. Историю конкретного клиента нельзя
считать пропущенной только потому, что в ней нет очередного платежа.

**Проверяем величину разрыва, а не место в рейтинге.** Для каждого этапа разрыв рассчитывается
вычитанием доли следующей оплаты у social из доли у остальных каналов вместе. Остальные
объединяются по числителям и знаменателям. Рабочий описательный критерий: разрыв
на первом переходе положителен и максимален, а средний разрыв на четырёх поздних
переходах не превышает половины первого. Порог 50% задаёт рабочее определение
слова «преимущественно» и не является тестом значимости; дополнительно показаны 25% и 75%.

Это условные переходы разных выборок, не воронка одной когорты. Результаты приводятся
также отдельно для обычных и начинавшихся с триала подписок. Одна подписка может
участвовать в нескольких этапах. Отсутствие оплаты не тождественно окончанию доступа.


In [ ]:
# Календарь нужен до августа: июньская оплата может иметь 35-дневное окно в августе.
renewal_calendar = q("""
select paid_at::date as day from prime.payments
where paid_at >= '2025-02-01'::timestamptz and paid_at < '2026-08-08'::timestamptz
group by 1 order by 1
""")
renewal_days = pd.DatetimeIndex(pd.to_datetime(renewal_calendar["day"]))
renewal_history_until = pd.to_datetime(
    q("select max(paid_at) as last_at from prime.payments").loc[0, "last_at"], utc=True)
assert pd.notna(renewal_history_until), "Нет истории платежей"

# Календарь уже загружен: определяем последний пропущенный день один раз,
# вместо вложенной проверки всей таблицы платежей для каждой строки.
import json

history_dates = pd.date_range("2025-02-01", "2026-08-07", freq="D")
history_missing = history_dates.difference(renewal_days)
last_missing = pd.Series(pd.NaT, index=history_dates, dtype="datetime64[ns]")
last_missing.loc[history_missing] = history_missing.to_numpy()
last_missing = last_missing.ffill()
index_calendar = [
    {"index_day": day.strftime("%Y-%m-%d"),
     "last_missing": None if pd.isna(last_missing.loc[day])
                     else last_missing.loc[day].strftime("%Y-%m-%d")}
    for day in pd.date_range("2026-01-01", "2026-06-30", freq="D")
]
index_calendar_json = json.dumps(index_calendar)

# У каждой подписки один канал владельца: расчёт по каналам сохраняет целую
# историю подписки. По месяцам до нумерации историю делить нельзя.
# ROW_NUMBER — cumcount() + 1, LEAD — shift(-1), LAG — shift(1).
# Календарь присоединяется ПОСЛЕ нумерации и поиска следующей оплаты.
RENEWAL_SQL = """
with ordered as (
    select p.payment_id, p.subscription_id, p.paid_at, p.amount,
           s.started_at, s.ended_at, s.is_trial,
           coalesce(c.acquisition_channel, 'без канала') as channel,
           lag(p.paid_at) over (
               partition by p.subscription_id, p.amount order by p.paid_at, p.payment_id
           ) as prev_same_amount
    from prime.payments p
    join prime.subscriptions s using(subscription_id)
    join prime.clients c on c.client_id = s.owner_client_id
    where p.status = 'success' and p.amount > 0
      and p.paid_at >= '2025-02-01'::timestamptz
      and p.paid_at < '2026-08-08'::timestamptz
      and p.paid_at >= s.started_at and (s.ended_at is null or p.paid_at < s.ended_at)
      and s.started_at >= '2025-02-01'::timestamptz
      and coalesce(c.acquisition_channel, 'без канала') = :channel
), clean as (
    select * from ordered
    where prev_same_amount is null or paid_at - prev_same_amount >= interval '20 minutes'
), numbered as (
    select *,
           row_number() over (partition by subscription_id order by paid_at, payment_id) as stage,
           lead(paid_at) over (partition by subscription_id order by paid_at, payment_id) as next_paid
    from clean
), index_calendar as (
    select * from jsonb_to_recordset(cast(:calendar_json as jsonb))
        as calendar(index_day date, last_missing date)
)
select d.index_day, n.channel, n.is_trial, n.stage,
       (d.last_missing is null or n.started_at::date > d.last_missing) as history_complete,
       count(*) as subscriptions,
       count(*) filter (where n.next_paid <= n.paid_at + interval '31 days') as next_31,
       count(*) filter (where n.next_paid <= n.paid_at + interval '35 days') as next_35,
       count(*) filter (where n.ended_at <= n.paid_at + interval '31 days') as ended_31,
       count(*) filter (
           where (n.next_paid is null or n.next_paid > n.paid_at + interval '31 days')
             and (n.ended_at is null or n.ended_at > n.paid_at + interval '31 days')
       ) as no_next_but_access
from numbered n
join index_calendar d on d.index_day = n.paid_at::date
where n.paid_at >= '2026-01-01'::timestamptz and n.paid_at < '2026-07-01'::timestamptz
  and n.stage <= 5
group by 1, 2, 3, 4, 5 order by 1, 2, 3, 4, 5
"""
renewal_parts = []
for channel in GROUPS:
    print(f"Продления: {channel}", flush=True)
    part = q(RENEWAL_SQL, channel=channel, calendar_json=index_calendar_json)
    assert part["channel"].eq(channel).all()
    renewal_parts.append(part)
renewal_daily = pd.concat(renewal_parts, ignore_index=True)
assert not renewal_daily.empty, "Нет событий продления за выбранный период"
renewal_daily["index_day"] = pd.to_datetime(renewal_daily["index_day"])
assert not renewal_daily.duplicated(["index_day", "channel", "is_trial", "stage", "history_complete"]).any()
assert renewal_daily["is_trial"].notna().all()

complete_renewal_days = []
for day in pd.date_range("2026-01-01", "2026-06-30", freq="D"):
    window = pd.date_range(day, day + pd.Timedelta(days=35), freq="D")
    # Требуем доступность всего последнего календарного дня окна.
    full_end = day.tz_localize("UTC") + pd.Timedelta(days=36)
    if window.difference(renewal_days).empty and full_end <= renewal_history_until:
        complete_renewal_days.append(day)
assert complete_renewal_days, "Нет полных окон для гипотезы 12"
renewal_history_sample = renewal_daily[renewal_daily["history_complete"]].copy()
renewal_sample = renewal_history_sample[
    renewal_history_sample["index_day"].isin(complete_renewal_days)].copy()
assert not renewal_sample.empty, "Нет событий с полной предысторией и последующим окном"
renewal_window_quality = renewal_daily.groupby("channel")["subscriptions"].sum().to_frame("До ограничения окон")
renewal_window_quality["Полная предыстория"] = renewal_history_sample.groupby("channel")["subscriptions"].sum()
renewal_window_quality["После ограничения окон"] = renewal_sample.groupby("channel")["subscriptions"].sum()
renewal_window_quality = renewal_window_quality.fillna(0)
renewal_window_quality["Охват, %"] = renewal_window_quality["После ограничения окон"] / renewal_window_quality["До ограничения окон"] * 100
display(renewal_window_quality.reindex(GROUPS))


In [ ]:
renewal_columns = ["subscriptions", "next_31", "next_35", "ended_31", "no_next_but_access"]
renewal_steps = renewal_sample.groupby(["channel", "is_trial", "stage"], as_index=False)[renewal_columns].sum()
assert renewal_steps["next_31"].le(renewal_steps["next_35"]).all()
assert renewal_steps["next_35"].le(renewal_steps["subscriptions"]).all()
assert renewal_steps["ended_31"].le(renewal_steps["subscriptions"]).all()
for outcome in ["next_31", "next_35", "ended_31", "no_next_but_access"]:
    renewal_steps[outcome + "_pct"] = renewal_steps[outcome] / renewal_steps["subscriptions"] * 100

renewal_all = renewal_steps.groupby(["channel", "stage"], as_index=False)[renewal_columns].sum()
for outcome in ["next_31", "next_35", "ended_31", "no_next_but_access"]:
    renewal_all[outcome + "_pct"] = renewal_all[outcome] / renewal_all["subscriptions"] * 100
renewal_rates = renewal_all.pivot(index="channel", columns="stage", values="next_31_pct").reindex(index=GROUPS, columns=[1, 2, 3, 4, 5])
renewal_counts = renewal_all.pivot(index="channel", columns="stage", values="subscriptions").reindex(index=GROUPS, columns=[1, 2, 3, 4, 5])
renewal_comparison = renewal_rates.copy()
renewal_comparison.columns = [f"{stage} - {stage + 1}, %" for stage in renewal_comparison.columns]
renewal_comparison["Подписок: 1 - 2"] = renewal_counts[1]
renewal_comparison["Подписок: 5 - 6"] = renewal_counts[5]
display(renewal_comparison)

# Отдельно показываем оба типа подписки и минимальный размер группы на этапах.
renewal_trial_rates = renewal_steps.pivot(index=["channel", "is_trial"], columns="stage", values="next_31_pct")
renewal_trial_counts = renewal_steps.pivot(index=["channel", "is_trial"], columns="stage", values="subscriptions")
renewal_trial_rates["Минимальная база этапа"] = renewal_trial_counts.min(axis=1)
display(renewal_trial_rates)
renewal_sensitivity = renewal_all.pivot(index="channel", columns="stage", values="next_35_pct").reindex(index=GROUPS, columns=[1, 2, 3, 4, 5]) - renewal_rates
display(Markdown(f"Максимальное изменение доли при расширении окна с 31 до 35 суток: {renewal_sensitivity.abs().max().max():.3f} п.п."))


pooled_renewal = renewal_all[~renewal_all["channel"].eq(FOCUS)].groupby("stage")[renewal_columns].sum()
renewal_gap = pd.DataFrame({
    "social, %": renewal_rates.loc[FOCUS],
    "Остальные, %": pooled_renewal["next_31"] / pooled_renewal["subscriptions"] * 100,
    "social, n": renewal_counts.loc[FOCUS],
    "Остальные, n": pooled_renewal["subscriptions"]
}).reindex([1, 2, 3, 4, 5])
renewal_gap["Разрыв, п.п."] = renewal_gap["Остальные, %"] - renewal_gap["social, %"]
display(renewal_gap)
gap_first = renewal_gap.loc[1, "Разрыв, п.п."]
gap_later = renewal_gap.loc[2:5, "Разрыв, п.п."].mean()
complete_comparison = renewal_gap.notna().all().all()
first_is_largest = complete_comparison and gap_first > 0 and gap_first >= renewal_gap.loc[2:5, "Разрыв, п.п."].max()
renewal_criterion_sensitivity = pd.DataFrame({"Допустимая доля позднего разрыва": [.25, .5, .75]})
renewal_criterion_sensitivity["Критерий выполнен"] = [
    bool(first_is_largest and gap_later <= threshold * gap_first)
    for threshold in renewal_criterion_sensitivity["Допустимая доля позднего разрыва"]]
display(renewal_criterion_sensitivity)
if not complete_comparison:
    hypothesis12_result = "Недостаточно сопоставимых этапов для проверки критерия."
else:
    supported = bool(first_is_largest and gap_later <= .5 * gap_first)
    hypothesis12_result = (
        f"Разрыв первого перехода {gap_first:+.2f} п.п.; средний разрыв поздних переходов "
        f"{gap_later:+.2f} п.п. Рабочий критерий концентрации на первом продлении "
        + ("выполнен. Это основание исследовать первый переход отдельно, но не доказательство эффекта меры."
           if supported else "не выполнен. Исключительный фокус на первом продлении этим критерием не обоснован.")
    )
display(Markdown("**Результат гипотезы 12.** " + hypothesis12_result))
display(renewal_all[["channel", "stage", "subscriptions", "next_31_pct", "ended_31_pct", "no_next_but_access_pct"]])
display(Markdown(
    "Отсутствие оплаты и окончание доступа — разные события. Таблица показывает оба показателя. "
    "next_31 и ended_31 могут пересекаться: оплата и окончание доступа способны произойти в одном окне; "
    "их нельзя автоматически складывать как непересекающиеся исходы."
))


### Дополнительная проверка: тарифный состав

Сравниваем доли тарифов и отток внутри одинаковых тарифов по всем каналам за
январь-июнь. Это обычная группировка, аналогичная другим разрезам.
**Ограничение:** без стандартизации нельзя количественно утверждать, какую часть
различий объясняет тарифный состав. Тариф взят из записи подписки, история смен
не восстановлена. Строки справочника свёрнуты до одного названия тарифа до соединения.

In [ ]:
tariff_names = q("""
select tariff_id, min(tariff_name) as tariff_name, count(distinct tariff_name) as names
from prime.tariffs group by 1 order by 1
""")
assert tariff_names["tariff_id"].is_unique
assert tariff_names["names"].eq(1).all(), "Неоднозначное название тарифа"
tariff_base = q("""
select m::date as month,
       coalesce(c.acquisition_channel, 'без канала') as comparison,
       s.tariff_id, count(*) as base_start,
       count(*) filter (where s.ended_at < m + interval '1 month') as churned
from generate_series('2026-01-01'::timestamptz, '2026-06-01'::timestamptz, interval '1 month') m
join prime.subscriptions s on s.started_at < m and (s.ended_at is null or s.ended_at >= m)
join prime.clients c on c.client_id = s.owner_client_id
group by 1, 2, 3 order by 1, 2, 3
""")
tariff_base["month"] = pd.to_datetime(tariff_base["month"])
assert not tariff_base.duplicated(["month", "comparison", "tariff_id"]).any()
tariff_base = tariff_base.merge(tariff_names[["tariff_id", "tariff_name"]], on="tariff_id", how="left", validate="m:1")
assert tariff_base["tariff_name"].notna().all()
check = tariff_base.groupby(["month", "comparison"])[["base_start", "churned"]].sum().sort_index()
assert check.equals(base.set_index(["month", "comparison"])[["base_start", "churned"]].sort_index())
tariff_base["churn_pct"] = tariff_base["churned"] / tariff_base["base_start"] * 100
tariff_base["share"] = tariff_base["base_start"] / tariff_base.groupby(["month", "comparison"])["base_start"].transform("sum")
tariff_summary = tariff_base.groupby(["tariff_name", "comparison"])[["base_start", "churned"]].sum()
tariff_summary["Доля базы, %"] = tariff_summary["base_start"] / tariff_summary.groupby("comparison")["base_start"].transform("sum") * 100
tariff_summary["Отток, %"] = tariff_summary["churned"] / tariff_summary["base_start"] * 100



tariff_rates = tariff_summary["Отток, %"].unstack("tariff_name").reindex(GROUPS)
tariff_shares = tariff_summary["Доля базы, %"].unstack("tariff_name").reindex(GROUPS)
display(tariff_rates)
display(tariff_shares)
evidence_tables["tariff_counts_and_rates"] = tariff_summary
display(tariff_summary)


### Дополнительная проверка: переоформления

Для завершений из начальной базы за январь-июнь ищем новое оформление того же
владельца в следующие 30 суток. Другую подписку соединяем обычным `LEFT JOIN`;
`MIN` оставляет дату первого нового оформления, чтобы события не размножались.
Учитываются только полные последующие окна; числа до и после ограничения видны
в таблице охвата. Отдельно проверяем число владельцев нескольких подписок.
**Ограничение:** отсутствие других подписок в доступной истории не доказывает
окончательного ухода человека. Переоформление не означает успешную оплату.

In [ ]:
subscription_dates = q("""
select started_at::date as day from prime.subscriptions
where started_at >= '2026-01-01'::timestamptz and started_at < '2026-08-01'::timestamptz
group by 1
""")
subscription_days = pd.DatetimeIndex(pd.to_datetime(subscription_dates["day"]))
return_missing = pd.date_range("2026-01-01", "2026-07-31", freq="D").difference(subscription_days)
last_subscription = q("select max(started_at) as last_at from prime.subscriptions").loc[0, "last_at"]
last_subscription = pd.to_datetime(last_subscription, utc=True)
assert pd.notna(last_subscription), "Нет истории оформлений"

return_events = q("""
with endings as (
    select s.subscription_id, s.owner_client_id, s.ended_at,
           coalesce(c.acquisition_channel, 'без канала') as channel
    from generate_series('2026-01-01'::timestamptz, '2026-06-01'::timestamptz, interval '1 month') m
    join prime.subscriptions s on s.started_at < m and s.ended_at >= m
                             and s.ended_at < m + interval '1 month'
    join prime.clients c on c.client_id = s.owner_client_id
)
select e.subscription_id, e.channel, e.ended_at, min(t.started_at) as new_at
from endings e
left join prime.subscriptions t on t.owner_client_id = e.owner_client_id
  and t.subscription_id <> e.subscription_id
  and t.started_at >= e.ended_at and t.started_at <= e.ended_at + interval '30 days'
group by 1, 2, 3
""")
assert return_events["subscription_id"].is_unique
assert return_events.groupby("channel").size().reindex(GROUPS, fill_value=0).eq(
    base.groupby("comparison")["churned"].sum().reindex(GROUPS)).all()
for column in ["ended_at", "new_at"]:
    return_events[column] = pd.to_datetime(return_events[column], utc=True)
return_events["window_end"] = return_events["ended_at"] + pd.Timedelta(days=30)
complete = return_events["window_end"].le(last_subscription)
for day in return_missing.tz_localize("UTC"):
    intersects = return_events["ended_at"].lt(day + pd.Timedelta(days=1)) & return_events["window_end"].ge(day)
    complete &= ~intersects
return_quality = pd.DataFrame({"Завершений всего": return_events.groupby("channel").size(),
    "Полных окон": return_events.loc[complete].groupby("channel").size()}).reindex(GROUPS).fillna(0)
return_events = return_events.loc[complete].copy()
return_events["returned_30"] = return_events["new_at"].notna()
return_summary = return_events.groupby("channel").agg(
    ended=("subscription_id", "size"), returned_30=("returned_30", "sum")).reindex(GROUPS)
return_summary["Переоформились за 30 суток, %"] = return_summary["returned_30"] / return_summary["ended"] * 100

ownership_structure = q("""
select coalesce(c.acquisition_channel, 'без канала') as channel,
       count(*) as owners, count(*) filter (where s.n > 1) as multiple_subscriptions
from (select owner_client_id, count(*) as n from prime.subscriptions
      where started_at < '2026-08-01'::timestamptz group by 1) s
join prime.clients c on c.client_id = s.owner_client_id
group by 1 order by 1
""")
display(ownership_structure)
display(return_quality)
display(return_summary)
multiple_total = int(ownership_structure["multiple_subscriptions"].sum())
display(Markdown(
    f"Владельцев более одной подписки в доступной истории до августа: {multiple_total}. "
    + ("Поэтому нулевые переоформления в этой структуре данных ожидаемы; "
       "проверка не позволяет заключить, что люди никогда не возвращаются в сервис."
       if multiple_total == 0 else "Возвраты характеризуют только новое оформление владельца в выбранном окне.")
))


### Итоги проверок

Ниже каждая гипотеза получает содержательный результат. Доли сопровождаются
числителями и знаменателями в соответствующих разделах; полные таблицы экспортируются.
Суммы начальных баз учитывают наблюдения подписок по месяцам, а не независимых людей.
Минимальный размер группы используется только как обзор, не как оценка точности.
Статистические тесты не выполняются. Для их добавления потребовалось бы учитывать
повторы по владельцам, множественные сравнения и заранее определить проверяемый эффект.


In [ ]:
group_sizes = pd.DataFrame({"Возраст подписки": age_counts.min(axis=1),
    "Возраст владельца": profile_counts["Возраст клиента"].min(axis=1),
    "Стаж владельца": profile_counts["Стаж клиента"].min(axis=1),
    "Активность": activity_counts.min(axis=1), "Участники": sharing_counts.min(axis=1),
    "Общие регионы": region_counts.loc[:, common_regions].min(axis=1),
    "Сервисы": service_counts.min(axis=1), "Поддержка": support_counts.min(axis=1)})
display(group_sizes)


hypothesis_rows = [
    {"Гипотеза": "1. Активная база", "Результат":
     f"Рост активной базы social между мартом и июнем 2026: {active_growth.iloc[:, 0].loc[FOCUS]:.2f}%. "
     "Вклад в рост выручки и сравнение с остальными рассчитываются в разделе 3.",
     "Ограничение": "Арифметическое объяснение не устанавливает причину"},
    {"Гипотеза": "2. Отток и накопление базы", "Результат":
     f"Отток social за март–июнь: {flow_main.loc[FOCUS, 'Отток, %']:.2f}% в месяц, "
     f"приток к начальной базе: {flow_main.loc[FOCUS, 'Новые оформления, %']:.2f}%. "
     "Высокий приток частично компенсирует завершения; точный баланс всего интервала — ниже.",
     "Ограничение": "Нужны и приток, и завершения; ранжирование оттока не равно ранжированию прироста"}
]
group_titles = {"3": "Возраст подписки", "4": "Возраст и стаж владельца", "5": "Активность",
                "7": "Совместное использование", "8": "Регион", "9": "Сервисы"}
for key, title in group_titles.items():
    hypothesis_rows.append({"Гипотеза": key + ". " + title, "Результат": group_findings[key],
        "Ограничение": "Отдельный разрез; состав не стандартизован; повторные наблюдения зависимы"})
for title, result, limit in [
    ("6. Причины отмены", reason_result, "Записанные причины, не установленный механизм"),
    ("10. Поддержка", support_result, "Малая группа; условный избыток не является эффектом меры"),
    ("11. Ошибки оплаты", payment_result, "Сырые группы; первая попытка периода, не всей истории"),
    ("12. Этап продления", hypothesis12_result, "Разные условные выборки; критерий описательный")]:
    hypothesis_rows.append({"Гипотеза": title, "Результат": result, "Ограничение": limit})
hypothesis_summary = pd.DataFrame(hypothesis_rows)
hypothesis_summary["Номер"] = hypothesis_summary["Гипотеза"].str.extract(r"^(\d+)").astype(int)
hypothesis_summary = hypothesis_summary.sort_values("Номер").drop(columns="Номер")
assert len(hypothesis_summary) == 12
# Markdown сохраняет длинные выводы без обрезания и чрезмерно широкой таблицы.
for row in hypothesis_summary.to_dict("records"):
    display(Markdown(f"**{row['Гипотеза']}.** {row['Результат']}\n\n*Ограничение:* {row['Ограничение']}"))


## 3. Факторный анализ выручки и движения базы

### 3.1. За счет чего изменилась дневная выручка каждого канала

Обозначения: R соответствует выручке, D числу календарных дней, A активной базе месяца,
P числу успешных платежей. Используется тождество:

`R / D = A * [P / (A * D)] * [R / P]`.

Три фактора: активная база, платежи на подписку в день и средний платеж.
Для расчёта используются цепные подстановки с усреднением вкладов по всем шести порядкам.
Факторы меняются по одному, для каждого шага определяется изменение произведения.
Усреднение распределяет взаимодействия факторов симметрично. Сумма вкладов
проверяется против фактического изменения дневной выручки.

На графике 3 показан waterfall изменения дневной выручки компании, в тысячах рублей в день. Для сравнения темпов каждый
вклад дополнительно делится на начальную дневную выручку соответствующего канала
и выражается в п.п. роста. Вклады разных каналов в п.п. не складываются
в общий рост компании.

Второй фактор отражает календарь оплат и длительность активности в месяце.
Основная выручка включает успешные платежи вне срока подписки, поэтому этот
фактор является нормировкой, а не частотой оплат только действующих подписок.
Средний платеж зависит от цен и состава платежей.
Декомпозиция объясняет изменение арифметически и не устанавливает причинность.


Каскад строится по приёму из С6: начало столбика равно накопленной сумме предыдущих
вкладов (`cumsum() - вклад`), отрицательный вклад идёт вниз, итог начинается от нуля.
Порядок показа столбиков не является новым порядком подстановок: на график попадают
уже усреднённые по шести порядкам вклады.

Для ответа на вопрос о различиях отдельно сравниваются social и все остальные.
Вклад остальных в рублях в день равен сумме вкладов отдельных каналов. Для выражения
в процентных пунктах эта сумма делится на их общую мартовскую дневную выручку. Это разложение
агрегированного роста через канальные вклады, а не новая мультипликативная модель
объединённого канала. Разность вкладов даёт разрыв темпов social минус остальные.


In [ ]:
factor_metrics = cash.merge(base, on=["month", "comparison"], validate="m:1")
factor_metrics["daily"] = factor_metrics["revenue"] / factor_metrics["month"].dt.days_in_month
factor_metrics["frequency"] = factor_metrics["payments"] / factor_metrics["active"] / factor_metrics["month"].dt.days_in_month
factor_metrics["average"] = factor_metrics["revenue"] / factor_metrics["payments"]
assert np.isclose(factor_metrics["active"] * factor_metrics["frequency"] * factor_metrics["average"], factor_metrics["daily"], atol=.001, rtol=0).all()
FACTOR_NAMES = {"active": "Активная база", "frequency": "Платежи на подписку в день", "average": "Средний платеж"}

def decompose(x0, x1):
    contributions = []
    for order in permutations(FACTOR_NAMES):
        current, effects = dict(x0), {}
        for factor in order:
            before = math.prod(current.values())
            current[factor] = x1[factor]
            effects[factor] = math.prod(current.values()) - before
        contributions.append(effects)
    result = pd.DataFrame(contributions).mean()
    assert np.isclose(result.sum(), math.prod(x1.values()) - math.prod(x0.values()), atol=0.00001, rtol=0)
    return result

decomp_rows, endpoints_rows = [], []
for a, b in PERIOD_PAIRS:
    label = f"{MONTHS[a.month]} - {MONTHS[b.month]}"
    for (policy, group), part in factor_metrics.groupby(["policy", "comparison"]):
        indexed = part.set_index("month")
        x0 = indexed.loc[a, list(FACTOR_NAMES)].astype(float).to_dict()
        x1 = indexed.loc[b, list(FACTOR_NAMES)].astype(float).to_dict()
        effects = decompose(x0, x1)
        initial = float(indexed.loc[a, "daily"])
        growth = (float(indexed.loc[b, "daily"]) / initial - 1) * 100
        assert np.isclose(effects.sum() / initial * 100, growth, atol=1e-9)
        endpoints_rows.append({"Период": label, "Платежи": policy, "Группа": group,
            "Начальная выручка в день": initial, "Конечная выручка в день": indexed.loc[b, "daily"],
            "Рост, %": growth, "Изменение базы, %": (x1["active"] / x0["active"] - 1) * 100,
            "Изменение частоты, %": (x1["frequency"] / x0["frequency"] - 1) * 100,
            "Изменение среднего платежа, %": (x1["average"] / x0["average"] - 1) * 100})
        for factor, value in effects.items():
            decomp_rows.append({"Период": label, "Платежи": policy, "Группа": group,
                "Фактор": FACTOR_NAMES[factor], "Вклад, руб. в день": value,
                "Вклад в рост, п.п.": value / initial * 100})

decomposition = pd.DataFrame(decomp_rows)
endpoints = pd.DataFrame(endpoints_rows)
main_decomposition = decomposition[decomposition["Платежи"].eq("Все успешные")]



factor_main = main_decomposition[main_decomposition["Период"].eq("Март - Июнь")]
factor_compact = factor_main.pivot(index="Группа", columns="Фактор", values="Вклад в рост, п.п.").reindex(GROUPS)
main_end = endpoints[endpoints["Период"].eq("Март - Июнь") & endpoints["Платежи"].eq("Все успешные")].set_index("Группа").reindex(GROUPS)
factor_compact["Рост дневной выручки, %"] = main_end["Рост, %"]
assert np.allclose(factor_compact[list(FACTOR_NAMES.values())].sum(axis=1), main_end["Рост, %"])
display(factor_compact)
factor_money = factor_main.pivot(index="Группа", columns="Фактор", values="Вклад, руб. в день").reindex(GROUPS)
factor_money["Фактическое изменение, руб. в день"] = main_end["Конечная выручка в день"] - main_end["Начальная выручка в день"]
assert np.allclose(factor_money[list(FACTOR_NAMES.values())].sum(axis=1), factor_money["Фактическое изменение, руб. в день"])
display(factor_money)


company_effects = factor_money[list(FACTOR_NAMES.values())].sum()
company_change = factor_money["Фактическое изменение, руб. в день"].sum()
assert np.isclose(company_effects.sum(), company_change, atol=.01, rtol=0)
company_growth = company_change / main_end["Начальная выручка в день"].sum() * 100
company_plot = company_effects.rename({"Активная база": "Активная\nбаза",
    "Платежи на подписку в день": "Платежи на подписку\nв день", "Средний платеж": "Средний\nплатёж"}) / 1000
waterfall(company_plot,
    f"3. Дневная выручка компании: {company_growth:+.2f}%; основной вклад — "
    + company_effects.abs().idxmax().lower(),
    subtitle="Март → июнь 2026, UTC. Сумма канальных вкладов; среднее шести порядков.\n"
             "Источник: успешные платежи и база подписок ПРАЙМ.")
plt.show()

others_mask = factor_money.index != FOCUS
other_initial = main_end.loc[main_end.index != FOCUS, "Начальная выручка в день"].sum()
other_final = main_end.loc[main_end.index != FOCUS, "Конечная выручка в день"].sum()
other_growth = (other_final / other_initial - 1) * 100
gap_factors = pd.DataFrame({
    "social, п.п.": factor_compact.loc[FOCUS, list(FACTOR_NAMES.values())],
    "Остальные, п.п.": factor_money.loc[others_mask, list(FACTOR_NAMES.values())].sum() / other_initial * 100
})
gap_factors["Разрыв social - остальные, п.п."] = gap_factors["social, п.п."] - gap_factors["Остальные, п.п."]
revenue_gap = main_end.loc[FOCUS, "Рост, %"] - other_growth
assert np.isclose(gap_factors["Разрыв social - остальные, п.п."].sum(), revenue_gap, atol=1e-9)
display(gap_factors)
factor_result = (
    f"Рост дневной выручки при сравнении марта и июня 2026: social: {main_end.loc[FOCUS, 'Рост, %']:+.2f}%; "
    f"остальные вместе: {other_growth:+.2f}%; разрыв {revenue_gap:+.2f} п.п. "
    + "; ".join(f"{name}: {value:+.2f} п.п." for name, value in
                gap_factors["Разрыв social - остальные, п.п."].items())
    + ". Это точное арифметическое разложение разрыва, а не оценка причинности."
)
display(Markdown("**Почему различаются темпы.** " + factor_result))

leading_factors = factor_compact[list(FACTOR_NAMES.values())].abs().idxmax(axis=1)
display(Markdown("**Ведущий по модулю фактор каждого канала:** " +
    "; ".join(f"{channel} - {factor}" for channel, factor in leading_factors.items()) + "."))


# Второй период - дополнительная проверка, без перебора всех пар каналов.
secondary_factors = main_decomposition[main_decomposition["Период"].eq("Январь - Июнь")].pivot(
    index="Группа", columns="Фактор", values="Вклад, руб. в день").reindex(GROUPS)
display(secondary_factors)


# Сравниваем social с суммарной выручкой остальных каналов при каждом правиле.
policy_gaps = []
for policy, part in endpoints[endpoints["Период"].eq("Март - Июнь")].groupby("Платежи"):
    focus_growth = part.loc[part["Группа"].eq(FOCUS), "Рост, %"].iloc[0]
    others = part.loc[~part["Группа"].eq(FOCUS)]
    others_growth = (others["Конечная выручка в день"].sum()
                     / others["Начальная выручка в день"].sum() - 1) * 100
    policy_gaps.append(focus_growth - others_growth)
robustness_result = (
    "При четырёх правилах учёта платежей разрыв роста дневной выручки "
    "social минус остальные каналы вместе при сравнении марта и июня 2026 "
    f"составляет от {min(policy_gaps):+.2f} до {max(policy_gaps):+.2f} п.п. "
    "Отставание social сохраняется при всех четырёх правилах."
)


### 3.2. Связь притока и оттока с активной базой марта и июня

Месячный баланс: конечная база = начальная + новые оформления − завершения
из начальной базы − завершения новых подписок. Июньская таблица остаётся отдельной
диагностикой; она не подменяет сравнение марта и июня.

Чтобы точно объяснить изменение активной базы между основными месяцами, используем:

`A_июнь − A_март = сумма(новые − отток − завершения новых) за март–май
                  + новые_июнь − новые_март`.

Основание: `A_месяца = B_начала + новые_месяца`, а изменение начальной базы
от марта до июня равно сумме месячных балансов марта, апреля и мая. Поэтому
июньский отток не уменьшает число подписок, пересекающих июнь: это не ошибка окна.
Каждый вклад делится на активную базу марта и выражается в п.п. её роста.

На графике 4 представлен waterfall для social. Таблица также сравнивает social с остальными
каналами, объединёнными через суммы числителей и баз. Такое разложение объясняет
наблюдаемый баланс, но не отвечает, сколько денег вернула бы мера удержания.


In [ ]:
base["ended_new"] = base["ended_subs"] - base["churned"]
assert base["ended_new"].ge(0).all()
base["ended_new_pct"] = base["ended_new"] / base["base_start"] * 100
base_contributions = base.set_index(["month", "comparison"])[["new_subs_pct", "churned_pct", "ended_new_pct"]].copy()
base_contributions.columns = ["Новые оформления", "Отток начальной базы", "Завершения новых подписок"]
base_contributions[["Отток начальной базы", "Завершения новых подписок"]] *= -1
base_contributions["Прирост базы, %"] = base.set_index(["month", "comparison"])["net_pct"]
assert np.allclose(base_contributions.iloc[:, :3].sum(axis=1), base_contributions["Прирост базы, %"])
june_contributions = base_contributions.xs(pd.Timestamp("2026-06-01"), level="month").reindex(GROUPS)
display(june_contributions)


march = base[base["month"].eq(pd.Timestamp("2026-03-01"))].set_index("comparison").reindex(GROUPS)
june = base[base["month"].eq(pd.Timestamp("2026-06-01"))].set_index("comparison").reindex(GROUPS)
intermediate = base[base["month"].between(pd.Timestamp("2026-03-01"), pd.Timestamp("2026-05-01"))]
flows = intermediate.groupby("comparison")[["new_subs", "churned", "ended_new"]].sum().reindex(GROUPS)
active_bridge_counts = pd.DataFrame({
    "Новые за март–май": flows["new_subs"],
    "Отток за март–май": -flows["churned"],
    "Завершения новых за март–май": -flows["ended_new"],
    "Новые июня минус новые марта": june["new_subs"] - march["new_subs"]
})
assert np.array_equal(active_bridge_counts.sum(axis=1).to_numpy(), (june["active"] - march["active"]).to_numpy())
active_bridge_pct = active_bridge_counts.div(march["active"], axis=0) * 100
assert np.allclose(active_bridge_pct.sum(axis=1), active_growth.iloc[:, 0].reindex(GROUPS))
active_bridge_comparison = pd.DataFrame({
    "social, п.п.": active_bridge_pct.loc[FOCUS],
    "Остальные, п.п.": active_bridge_counts.drop(index=FOCUS).sum() / march.drop(index=FOCUS)["active"].sum() * 100
})
active_bridge_comparison["Разрыв, п.п."] = active_bridge_comparison["social, п.п."] - active_bridge_comparison["Остальные, п.п."]
display(active_bridge_counts)
display(active_bridge_pct)
display(active_bridge_comparison)
bridge_labels = {"Новые за март–май": "Новые\nмарт–май", "Отток за март–май": "Отток\nмарт–май",
    "Завершения новых за март–май": "Завершения\nновых", "Новые июня минус новые марта": "Новые июня\nминус новые марта"}
waterfall(active_bridge_pct.loc[FOCUS].rename(bridge_labels),
    f"4. Активная база social: {active_bridge_pct.loc[FOCUS].sum():+.2f}% между мартом и июнем",
    ylabel="Вклад в рост активной базы, п.п.",
    subtitle="Знаменатель: активная база марта. Точный баланс марта–июня.\nИсточник: prime.subscriptions, UTC.")
plt.show()
base_result = (
    f"Рост активной базы между мартом и июнем 2026: social {active_bridge_comparison['social, п.п.'].sum():+.2f}%, "
    f"остальных {active_bridge_comparison['Остальные, п.п.'].sum():+.2f}%. "
    "Разрыв складывается из: " + "; ".join(
        f"{name}: {value:+.2f} п.п." for name, value in active_bridge_comparison["Разрыв, п.п."].items())
    + ". Приток и завершения оцениваются совместно; денежный эффект изменения оттока здесь не моделируется."
)
display(Markdown("**Связь с ростом базы.** " + base_result))


## 4. Сценарный анализ удержания

Для каждого канала снижаем июньский отток на 1 п.п. при неизменной начальной базе
и прочих равных. Снижение ограничивается фактическим оттоком, чтобы результат
не стал отрицательным. Дополнительно сохранённые подписки = начальная база *
снижение в п.п. / 100. Дробное значение отражает условный расчётный объём и не обозначает число
фактически установленных событий.

Одинаковое снижение в п.п. не означает одинаковых затрат и достижимости.
Крупный канал может иметь больший абсолютный потенциал при меньшем оттоке.
Сценарий не является прогнозом или оценкой причинного эффекта меры.
Денежный эффект и окупаемость не рассчитываются без данных о будущих оплатах,
сроке удержания, затратах и марже.

Поскольку при оттоке выше 1% сохранение равно 1% начальной базы, ранжирование этого
сценария повторяет размер базы. Оно не выбирает самый выгодный канал. Дополнительно
показывается относительное снижение оттока: один п.п., делённый на фактический отток.
Без затрат и оценки достижимости одинаковые п.п. не являются сопоставимыми мерами.
Пятый график посвящён разрыву продлений; сценарий остаётся таблицей.


In [ ]:
june_base=base[base['month'].eq(pd.Timestamp('2026-06-01'))].set_index('comparison').reindex(GROUPS)
scenario=pd.DataFrame({'Начальная база':june_base.base_start,'Отток, %':june_base.churned_pct})
scenario['Снижение, п.п.']=np.minimum(scenario['Отток, %'],1.)
scenario['Сценарный отток, %']=scenario['Отток, %']-scenario['Снижение, п.п.']
scenario['Дополнительно сохранены']=scenario['Начальная база']*scenario['Снижение, п.п.']/100
scenario['Сохранены из начальной базы']=june_base.base_start-june_base.churned+scenario['Дополнительно сохранены']
assert scenario['Сценарный отток, %'].ge(0).all()
assert scenario['Сохранены из начальной базы'].le(scenario['Начальная база']).all()

scenario["Относительное снижение оттока, %"] = (
    scenario["Снижение, п.п."] / scenario["Отток, %"].replace(0, np.nan) * 100)
display(scenario)
fig, ax = plt.subplots(figsize=(10, 4.5), layout="constrained")
ax.plot(renewal_gap.index, renewal_gap["Разрыв, п.п."], marker="o", color="#F28E2B")
for stage, value in renewal_gap["Разрыв, п.п."].items():
    if pd.notna(value):
        ax.annotate(f"{value:+.2f}", (stage, value), xytext=(0, 8), textcoords="offset points", ha="center")
ax.axhline(0, color="#6B7280", linewidth=.8)
ax.set_xticks([1, 2, 3, 4, 5], ["1→2", "2→3", "3→4", "4→5", "5→6"])
ax.set(title=("5. Разрыв продлений: первый переход " + f"{gap_first:+.2f} п.п., поздние в среднем {gap_later:+.2f} п.п."),
       ylabel="Доля следующей оплаты: остальные минус social, п.п.",
       xlabel="Переход между успешными оплатами; 31 сутки, полные 35-дневные окна и предыстория\n"
              "Январь–июнь 2026, UTC; разные условные выборки, не одна когорта")
ax.grid(axis="y", alpha=.2)
ax.margins(y=.25)
plt.show()


## 5. Выводы и бизнес-рекомендация

Итоговая таблица сравнивает все каналы. Рост выручки и факторные вклады относятся
к сравнению марта и июня. Основной показатель оттока рассчитан за март-июнь,
дополнительный рассчитан за январь–июнь. Причины отмен также относятся к январю–июню 2026. Сценарий удержания относится только к июню.
Переходы между оплатами оценены по событиям января-июня с полными последующими
окнами. Периоды и знаменатели указаны в названиях показателей и описании методов.

In [ ]:
summary = factor_compact.copy()
summary = summary.rename(columns={
    "Рост дневной выручки, %": "Рост дневной выручки: март - июнь, %",
    "Активная база": "Вклад базы: март - июнь, п.п.",
    "Платежи на подписку в день": "Вклад оплат: март - июнь, п.п.",
    "Средний платеж": "Вклад платежа: март - июнь, п.п."
})
summary["Рост активной базы: март - июнь, %"] = active_growth.iloc[:, 0]
summary["Взвешенный месячный отток: март-июнь, %"] = flow_main["Отток, %"]
summary["Взвешенный месячный отток: январь-июнь, %"] = weighted_churn["Отток, %"]
summary["Сценарий июня: сохранено дополнительно"] = scenario["Дополнительно сохранены"]
display(summary.sort_values("Рост дневной выручки: март - июнь, %"))

conclusion_text = (
    "**Что произошло.** " + factor_result + "\n\n"
    "**Устойчивость.** " + robustness_result + "\n\n"
    "**Как менялась база.** " + base_result + "\n\n"
    "**Что дали проверки.** " + reason_result + ". " + support_result + " " + payment_result + "\n\n"
    "**Продления.** " + hypothesis12_result + "\n\n"
    "**Ограничения.** Причинный механизм не установлен. Отдельные разрезы не выравнивают состав. "
    "Дни с событиями не гарантируют полной загрузки, а отсутствие оплаты не означает окончание доступа.\n\n"
    "**Следующий шаг.** В первые две недели аналитик и владелец данных уточняют аномальные платежи "
    "и разрыв выгрузки. CRM и продукт сопоставляют ушедших и оставшихся social того же тарифа, "
    "близкого возраста подписки и календарного месяца; используют причины отмен и интервью. "
    "Меру удержания выбирают после этой диагностики и проверяют случайным назначением по владельцу. "
    "Заранее фиксируют полезный эффект, размер выборки, затраты и окно 30/60 дней. "
    "Масштабирование — при положительной экономике и приемлемых возвратах и обращениях.\n\n"
    "**Сценарий.** Снижение июньского оттока social на 1 п.п. соответствует "
    f"{scenario.loc[FOCUS, 'Дополнительно сохранены']:.2f} подписки и "
    f"{scenario.loc[FOCUS, 'Относительное снижение оттока, %']:.1f}% относительного снижения оттока. "
    "Это иллюстрация размера базы, не прогноз эффекта или оценка окупаемости."
)
display(Markdown(conclusion_text))


### Рекомендация и критерий решения

1. **Аналитик и владелец данных:** уточнить смысл успешных платежей после завершения
   подписки и источник апрельского разрыва. Эти вопросы влияют на трактовку оплат
   и поведенческих признаков.
2. **CRM и продукт, первые две недели:** исследовать причины ухода в канале
   social, выбранном для подробной диагностики. Сопоставить завершивших и сохранивших
   подписку владельцев того же канала, тарифа, календарного месяца и близкого возраста подписки.
   Использовать причины отмен и целевые интервью; запросить время отключения
   автопродления и причины отказов платёжного шлюза. Фокус на конкретном этапе
   выбирается по рассчитанным величинам разрыва продлений, а не по рангу social.
   Если критерий концентрации разрыва на первом переходе не выполнен, исследование
   охватывает несколько этапов. Конкретная мера и её эффект требуют эксперимента.
3. **CRM, продукт и аналитик:** выбранную меру проверить случайным назначением
   по владельцу, со стратификацией по каналу и тарифу. Сначала зафиксировать
   минимально полезный эффект и необходимый размер выборки; выдержать полное окно
   наблюдения. Ориентир 1 п.п. в сценарии не заменяет расчёт мощности.
   Оценивать сохранение платящих владельцев через 30/60 дней и
   дополнительный доход за вычетом затрат. Дополнительно контролировать возвраты и обращения.
4. **Руководитель:** выбирать масштабирование по эффекту и экономике эксперимента.
   Размер канала учитывать отдельно от достижимости снижения оттока и стоимости меры. Текущих данных недостаточно для обоснования массовой скидки или
   перераспределения рекламного бюджета.

### Условия интерпретации

- Календарная нормировка не устраняет сезонность, цены, состав клиентов или
  неполноту данных. Обнаружение дня с событиями не гарантирует полноту выгрузки.
- Показатели повторных месячных наблюдений подписок не равны долям уникальных
  владельцев за полугодие. Активная неоплаченная подписка не равна платящему удержанию.
- Возрастные, региональные, поведенческие и другие разрезы проверены отдельно;
  совместные и ненаблюдаемые факторы не устранены. Сервисы могут пересекаться.
- Стандартизация не выполняется: состав каналов и групп оплаты не выровнен.
  Малые группы и неполные окна ограничивают вывод; статистическая неопределённость
  не оценена. Сравнения внутри групп сами по себе не измеряют вклад состава.
- Переходы между оплатами рассчитаны на разных условных выборках с полными окнами.
  Их нельзя перемножать как воронку одной когорты. Начисление очередной оплаты
  и сохранение доступа учитываются как разные результаты.
- Отток не переведён в деньги без модели. Сценарий не прогнозирует результат меры.


### Сохранение результатов и комплект сдачи

После выполнения таблицы и сформированная записка сохраняются в **новую папку
запуска** внутри `reports/project1_revised`. В неё попадают только результаты этой
версии; старые файлы из `reports/project1_final` не читаются и не перезаписываются.
Сохраняются числители, знаменатели, сравнения, аудит, конечные значения факторов
и список выгрузок. Клиентские идентификаторы не экспортируются.

Код ниже формирует краткую записку из новых результатов. Её размещение в корне
под требуемым именем и проверка объёма одной страницы относятся к подготовке сдачи.
Итоговый ноутбук должен называться `notebooks/project1.ipynb`; эта отдельная версия
сама его не заменяет и не меняет Git-ветку.

Декларация также должна соответствовать этой версии: переработка с помощью
ChatGPT/Codex, SQL, проверки полноты истории, факторные разложения и waterfall.
В ноутбуке присутствуют сохранённые результаты выполнения. Самостоятельные
проверки и уровень понимания автор указывает по фактически выполненной работе. Старую декларацию о стандартизации использовать нельзя:
в этой работе стандартизация не выполняется.


In [ ]:
from datetime import datetime, timezone
import json

run_id = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
report_dir = ROOT / "reports" / "project1_revised" / run_id
report_dir.mkdir(parents=True, exist_ok=False)
exports = {
    "audit": audit, "payment_calendar": payment_calendar,
    "tariff_history": tariff_history, "tariff_overlaps": tariff_overlaps,
    "duplicate_summary": duplicate_summary,
    "duplicates_by_channel": duplicate_by_channel.reset_index(),
    "growth": revenue_comparison.rename_axis("channel").reset_index(),
    "sensitivity": sensitivity, "movement": base,
    "base_contributions": base_contributions.reset_index(),
    "active_bridge_counts": active_bridge_counts.reset_index(),
    "active_bridge_pct": active_bridge_pct.reset_index(),
    "active_bridge_comparison": active_bridge_comparison.rename_axis("factor").reset_index(),
    "hypothesis_summary": hypothesis_summary,
    "renewal_steps": renewal_steps, "renewal_all": renewal_all,
    "renewal_comparison": renewal_comparison.reset_index(),
    "renewal_gap": renewal_gap.rename_axis("stage").reset_index(),
    "renewal_criterion_sensitivity": renewal_criterion_sensitivity,
    "renewal_window_coverage": renewal_window_quality.reset_index(),
    "renewal_horizon_sensitivity": renewal_sensitivity.reset_index(),
    "group_sizes": group_sizes.reset_index(), "reasons": reason_summary,
    "region_coverage": region_coverage.reset_index(),
    "support": support_impact.reset_index(), "support_coverage": support_coverage,
    "activity_coverage": activity_coverage, "tariff_shares": tariff_shares.reset_index(),
    "returns": return_summary.reset_index(), "return_coverage": return_quality.reset_index(),
    "ownership_structure": ownership_structure,
    "payments": payment_comparison.reset_index(), "payment_frequency": payment_frequency.reset_index(),
    "payment_cohort_coverage": payment_cohort_coverage.reset_index(),
    "payment_counts": payment_counts.reset_index(),
    "cash": cash, "factor_endpoints": endpoints, "factor_metrics": factor_metrics,
    "decomposition": decomposition, "gap_factors": gap_factors.rename_axis("factor").reset_index(),
    "scenario": scenario.reset_index(), "summary": summary.reset_index()
}
exports.update({name: table.reset_index() for name, table in evidence_tables.items()})
exports.update({f"comparison_{name}": table.reset_index() for name, table in group_comparisons.items()})
for name, table in exports.items():
    table.to_csv(report_dir / f"{name}.csv", index=False, encoding="utf-8-sig")
(report_dir / "conclusion.md").write_text(conclusion_text + "\n", encoding="utf-8")

memo_text = (
    "# ПРАЙМ: рост выручки и удержание по каналам\n\nШевченко Андрей, 262.\n\n"
    "**Вопрос.** Почему различаются темпы роста дневной выручки марта–июня 2026 и что исследовать первым?\n\n"
    "**Результат.** " + factor_result + "\n\n"
    "**База.** " + base_result + "\n\n"
    "**Проверки.** " + hypothesis12_result + " Записанные причины: " + reason_result + ".\n\n"
    "**Надёжность.** Проверены баланс базы и сумма вкладов. " + robustness_result + " "
    "Апрельский разрыв исключён из конечных точек и соответствующих событийных окон; "
    "номер оплаты проверяется также по предшествующей истории. Полнота источника и причины аномалий требуют подтверждения. "
    "Состав групп не стандартизован, причинный эффект не установлен.\n\n"
    "**Решение.** Первые две недели — проверить качество данных и исследовать причины ухода social "
    "в сопоставимых группах тарифа, возраста подписки и месяца. Затем выбрать одну меру и проверить "
    "случайным назначением по владельцу: платящее удержание 30/60 дней, доход за вычетом затрат, возвраты и обращения. "
    "Массовые скидки и перераспределение бюджета пока не обоснованы.\n\n"
    f"**Масштаб.** Минус 1 п.п. июньского оттока social — условно {scenario.loc[FOCUS, 'Дополнительно сохранены']:.0f} "
    "сохранённых подписок, без оценки достижимости и денежного эффекта.\n\n"
    "Источник: project1_revised.ipynb, успешные списания, UTC."
)
(report_dir / "ЗАПИСКА-ПРОЕКТ-1.md").write_text(memo_text + "\n", encoding="utf-8")
manifest = {"notebook": "notebooks/project1_revised.ipynb", "run_utc": run_id,
            "tables": sorted(exports), "pandas": pd.__version__, "numpy": np.__version__,
            "python": sys.version, "timezone": "UTC"}
(report_dir / "manifest.json").write_text(json.dumps(manifest, ensure_ascii=False, indent=2), encoding="utf-8")
print(f"Создан новый каталог результатов: {report_dir}")


In [ ]:
# Отдельное исследование: отток social при одинаковом составе групп.
# Использует функцию q() и таблицу base из project1_revised.
# Ноутбук и файлы на диске не изменяет.

import numpy as np
import pandas as pd
from IPython.display import display, Markdown

matched_data = q("""
    with subscription_months as (
        select
            m::date as month,
            coalesce(c.acquisition_channel, 'без канала') as channel,
            coalesce(s.tariff_id::text, 'Тариф не указан') as tariff,
            floor(
                extract(epoch from (m - s.started_at)) / 86400
            )::int as age_days,
            (s.ended_at < m + interval '1 month') as churned
        from generate_series(
            '2026-01-01'::timestamptz,
            '2026-06-01'::timestamptz,
            interval '1 month'
        ) m
        join prime.subscriptions s
          on s.started_at < m
         and (s.ended_at is null or s.ended_at >= m)
        join prime.clients c
          on c.client_id = s.owner_client_id
    )
    select
        month,
        channel,
        tariff,
        case
            when age_days < 30 then '0-29 дней'
            when age_days < 90 then '30-89 дней'
            when age_days < 180 then '90-179 дней'
            when age_days < 365 then '180-364 дня'
            else '365 дней и более'
        end as age_group,
        (age_days / 30) as age_30d,
        count(*) as base_start,
        count(*) filter (where churned) as churned
    from subscription_months
    group by 1, 2, 3, 4, 5
    order by 1, 2, 3, 4, 5
""")

matched_data["month"] = pd.to_datetime(
    matched_data["month"]
).astype("datetime64[ns]")

assert not matched_data.duplicated(
    ["month", "channel", "tariff", "age_group", "age_30d"]
).any()
assert matched_data["base_start"].gt(0).all()
assert matched_data["churned"].between(
    0, matched_data["base_start"]
).all()

# Новый разрез должен сохранить базу и число завершений из проекта.
matched_actual = (
    matched_data.groupby(["month", "channel"])[["base_start", "churned"]]
    .sum()
    .sort_index()
)
matched_expected = base[
    ["month", "comparison", "base_start", "churned"]
].rename(columns={"comparison": "channel"}).copy()
matched_expected["month"] = pd.to_datetime(
    matched_expected["month"]
).astype("datetime64[ns]")
matched_expected = matched_expected.set_index(
    ["month", "channel"]
).sort_index()

pd.testing.assert_frame_equal(
    matched_actual, matched_expected, check_dtype=False
)

matched_data["comparison_group"] = np.where(
    matched_data["channel"].eq("social"), "social", "Остальные"
)


def compare_matched_churn(data, group_columns, min_base=1):
    """Сравнить отток при одинаковых весах сопоставимых групп."""
    aggregated = data.groupby(
        group_columns + ["comparison_group"]
    )[["base_start", "churned"]].sum()

    counts = (
        aggregated["base_start"]
        .unstack("comparison_group", fill_value=0)
        .reindex(columns=["social", "Остальные"], fill_value=0)
    )
    events = (
        aggregated["churned"]
        .unstack("comparison_group", fill_value=0)
        .reindex(columns=["social", "Остальные"], fill_value=0)
    )

    # Оставляем группы с достаточной базой в обеих частях сравнения.
    eligible = counts.ge(min_base).all(axis=1)
    common_counts = counts.loc[eligible]
    common_events = events.loc[eligible]

    if common_counts.empty:
        raise ValueError(
            f"Нет сопоставимых групп при минимальной базе {min_base}."
        )

    total_counts = counts.sum()
    raw_rates = events.sum() / total_counts * 100
    common_rates = common_events / common_counts * 100
    common_raw_rates = common_events.sum() / common_counts.sum() * 100

    # Основной вариант: общий состав social и остальных.
    # Каждой группе даем один и тот же вес для обоих каналов.
    weights = common_counts.sum(axis=1)
    weights = weights / weights.sum()

    adjusted_rates = common_rates.mul(weights, axis=0).sum()

    assert np.isclose(weights.sum(), 1)
    assert adjusted_rates.between(0, 100).all()

    coverage = common_counts.sum() / total_counts * 100
    raw_gap = raw_rates["social"] - raw_rates["Остальные"]
    common_gap = (
        common_raw_rates["social"] - common_raw_rates["Остальные"]
    )
    adjusted_gap = adjusted_rates["social"] - adjusted_rates["Остальные"]

    result = {
        "Отток social до отбора, %": raw_rates["social"],
        "Отток остальных до отбора, %": raw_rates["Остальные"],
        "Исходный разрыв, п.п.": raw_gap,
        "Разрыв после отбора общих групп, п.п.": common_gap,
        "Отток social при общем составе, %": adjusted_rates["social"],
        "Отток остальных при общем составе, %": adjusted_rates["Остальные"],
        "Разрыв при общем составе, п.п.": adjusted_gap,
        "Изменение разрыва после выравнивания, п.п.": adjusted_gap - common_gap,
        "Охват базы social, %": coverage["social"],
        "Охват базы остальных, %": coverage["Остальные"],
        "Всего групп": len(counts),
        "Сопоставимых групп": len(common_counts),
        "Групп с большим оттоком social": int(
            common_rates["social"].gt(common_rates["Остальные"]).sum()
        ),
    }

    # Проверяем, зависит ли вывод от выбора общей структуры.
    for target in ["social", "Остальные"]:
        target_weights = common_counts[target] / common_counts[target].sum()
        target_rates = common_rates.mul(target_weights, axis=0).sum()
        result[f"Разрыв при составе {target}, п.п."] = (
            target_rates["social"] - target_rates["Остальные"]
        )

    details = pd.DataFrame({
        "База social": common_counts["social"],
        "Завершения social": common_events["social"],
        "Отток social, %": common_rates["social"],
        "База остальных": common_counts["Остальные"],
        "Завершения остальных": common_events["Остальные"],
        "Отток остальных, %": common_rates["Остальные"],
        "Разрыв, п.п.": common_rates["social"] - common_rates["Остальные"],
        "Общий вес": weights,
    })
    details["Вклад в общий разрыв, п.п."] = (
        details["Разрыв, п.п."] * details["Общий вес"]
    )
    assert np.isclose(
        details["Вклад в общий разрыв, п.п."].sum(), adjusted_gap
    )

    return result, details


matched_results = []
matched_details = {}

# Основной период соответствует анализу роста выручки.
# Полугодие используем как дополнительную проверку.
for period_name, period_data in [
    ("Март–июнь 2026", matched_data[matched_data["month"].dt.month.ge(3)]),
    ("Январь–июнь 2026", matched_data),
]:
    specifications = [
        ("Только месяц", ["month"], 1),
        ("Месяц и тариф", ["month", "tariff"], 1),
        ("Месяц и возраст", ["month", "age_group"], 1),
        ("Месяц, тариф и возраст", ["month", "tariff", "age_group"], 1),
        (
            "Месяц, тариф и возраст; база от 30",
            ["month", "tariff", "age_group"],
            30,
        ),
        (
            "Месяц, тариф и возраст; база от 100",
            ["month", "tariff", "age_group"],
            100,
        ),
        (
            "Месяц, тариф и возраст с шагом 30 дней",
            ["month", "tariff", "age_30d"],
            1,
        ),
    ]

    for specification, columns, minimum in specifications:
        result, details = compare_matched_churn(
            period_data, columns, min_base=minimum
        )
        matched_results.append({
            "Период": period_name,
            "Сравнение": specification,
            **result,
        })
        matched_details[(period_name, specification)] = details

matched_summary = pd.DataFrame(matched_results)

display(Markdown("### Сравнение оттока при одинаковом составе групп"))
display(matched_summary[[
    "Период",
    "Сравнение",
    "Исходный разрыв, п.п.",
    "Разрыв после отбора общих групп, п.п.",
    "Разрыв при общем составе, п.п.",
    "Охват базы social, %",
    "Охват базы остальных, %",
    "Сопоставимых групп",
]].round(3))

matched_main = matched_summary.loc[
    matched_summary["Период"].eq("Март–июнь 2026")
    & matched_summary["Сравнение"].eq("Месяц, тариф и возраст")
].iloc[0]

display(Markdown(
    f"За период с марта по июнь исходный разрыв оттока social и остальных "
    f"составил {matched_main['Исходный разрыв, п.п.']:+.3f} п.п. "
    f"После отбора общих групп он составил "
    f"{matched_main['Разрыв после отбора общих групп, п.п.']:+.3f} п.п., "
    f"а при одинаковом составе по месяцу, тарифу и возрасту подписки "
    f"{matched_main['Разрыв при общем составе, п.п.']:+.3f} п.п.\n\n"
    f"В сравнение вошло {matched_main['Охват базы social, %']:.2f}% "
    f"наблюдений social и {matched_main['Охват базы остальных, %']:.2f}% "
    f"наблюдений остальных каналов."
))

display(Markdown("### Проверка выбора весов"))
display(matched_summary.loc[
    matched_summary["Сравнение"].eq("Месяц, тариф и возраст"),
    [
        "Период",
        "Разрыв при общем составе, п.п.",
        "Разрыв при составе social, п.п.",
        "Разрыв при составе Остальные, п.п.",
    ],
].round(3))

display(Markdown("### Численность и отток в сопоставимых группах"))
display(
    matched_details[("Март–июнь 2026", "Месяц, тариф и возраст")].round(3)
)

display(Markdown(
    "**Как читать результат.** Положительный разрыв означает, что отток "
    "social выше. Сокращение разрыва после выравнивания показывает, что "
    "часть исходного различия связана с составом рассмотренных групп. "
    "Если разрыв сохраняется, месяц, тариф и выбранные интервалы возраста "
    "не объясняют его полностью.\n\n"
    "**Ограничения.** В базе учитываются наблюдения подписок по месяцам, "
    "поэтому одна подписка может встречаться несколько раз. "
    "Сравнение описательное, статистическая значимость не проверяется. "
    "Возраст внутри интервалов и другие признаки могут различаться. "
    "Используется тариф из записи подписки; история смены тарифов "
    "не восстанавливается. Результат не доказывает причинное влияние "
    "канала привлечения на уход."
))